# Capstone Visualization — All Charts
Resource Curse & Economic Complexity | Moody's Ratings Capstone

Run this notebook top-to-bottom to regenerate every chart in `Final/charts/`.

**Prerequisites:** All intermediate files in `intermediary/` must exist (run NB1–NB4 first).

| Section | Charts | Source |
|---------|--------|--------|
| 1. Clustering (NB4) | 01, 02, 03, 04, 05 | NB4 code (inline) |
| 2. ML Training (NB5) | CSV outputs for charts 07–13 | run_nb5.py |
| 3. ML Charts | 07, 08, 09, 10, 11, 13 | viz_updates.py |
| 4. Regressions (NB6) | 14, 15, 16, 17 + diagnostics | run_nb6.py + viz_updates.py |
| 5. Diagnostics | 26, 27, 28, 29, 30, 31, 32, 33, 34 | viz_updates.py |
| 6. Case Studies | 21, 22, 23, 24 | See note in section |

In [ ]:
import os, sys, warnings

# ── Find and cd to project root (FINAL CODE RECAP/) ──────────────────────────
_d = os.getcwd()
for _ in range(8):
    if os.path.basename(_d) == 'FINAL CODE RECAP':
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
print("Working dir:", os.getcwd())

# ── Add scripts/ to path ──────────────────────────────────────────────────────
_scripts = os.path.join(os.getcwd(), 'scripts')
if _scripts not in sys.path:
    sys.path.insert(0, _scripts)

warnings.filterwarnings('ignore')

# ── Standard imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import scipy.stats as stats
from types import SimpleNamespace

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

try:
    import shap
    HAS_SHAP = True
    print("SHAP available")
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed")

# ── Import shared style utilities from viz_utils ─────────────────────────────
from viz_utils import (
    CLUSTER_LABELS, CLUSTER_COLORS, INCLUDE_LIST,
    WRITE_CONFIG, BG, GRID, FONT, NAVY, PALETTE,
    load_master, load_master_wide, load_clusters, load_nr,
    shorten_feat, base_layout, save,
    analyze_country_missingness, analyze_variable_missingness,
    build_sample,
)

# ── Output directories ────────────────────────────────────────────────────────
OUT        = 'Final/NB5'
OUT_NB6    = 'Final/NB6'
OUT_CHARTS = 'Final/charts'
os.makedirs(OUT, exist_ok=True)
os.makedirs(OUT_NB6, exist_ok=True)
os.makedirs(OUT_CHARTS, exist_ok=True)

# ── Shared Plotly style ───────────────────────────────────────────────────────
STYLE = {
    'font_family':       'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size':         11,
    'axis_title_size':   13,
    'legend_size':       11,
    'annotation_size':   11,
    'title_color':       '#1a2744',
    'template':          'plotly_white',
    'plot_bg':           '#fafafa',
    'paper_bg':          '#fafafa',
    'chart_height':      550,
    'chart_height_small':420,
    'chart_height_tall': 700,
    'margin':            dict(l=60,  r=40,  t=10, b=50),
    'margin_bar':        dict(l=160, r=130, t=10, b=50),
    'grid_color':        '#e5e7eb',
    'grid_width':        0.5,
    'zero_line_color':   '#c9cfd6',
}

NB5 = OUT  # alias for viz_updates compatibility

plt.rcParams.update({
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
})

print("Setup complete.")

## Section 1 — Clustering (NB4): Charts 01–05

Charts 01 (sample map), 02 (variable correlations), 03 (PCA biplot), 04 (cluster world maps), 05 (animated Rosling chart).

This section replicates the full NB4 clustering pipeline: data loading, PCA, K-means, and chart output.

In [ ]:
# # Clustering: Natural Resource Profiles
# 
# **Capstone Project — Moody's Ratings**  
# *Pipeline step 4 of 5: Identifying resource-dependency typologies*
# 
# This notebook classifies countries into four natural resource profile groups using PCA and K-Means clustering on per-capita production values. The clustering is run for three time windows to capture both cross-sectional structure and temporal shifts.
# 
# **Methodology:**
# 1. Production values (quantity x price, in USD) are divided by population to obtain per-capita figures
# 2. A log(1+x) transformation compresses the extreme right skew typical of resource data
# 3. PCA reduces the feature space to 2 components (PC1 captures hydrocarbons, PC2 captures minerals)
# 4. K-Means (k=4) partitions countries in PCA space into four groups
# 
# **Why these choices:**
# - *Per capita* (not per GDP): avoids mixing resource profiles with economic structure, which is what we're trying to predict (ECI)
# - *log1p* (not z-scoring): appropriate for distributions with 1000:1 ratios between smallest and largest producers; StandardScaler would compress most countries into a narrow band
# - *PCA then K-Means* (not K-Means on raw features): removes correlated noise (oil and gas co-occur), reduces dimensionality for better Euclidean distance behaviour
# - *k=4*: validated with silhouette analysis; produces a clean typology matching the economic literature
# 
# **Three variants:**
# - **1995 snapshot:** Cluster assignments based on 1995 production profiles (baseline year)
# - **2019 snapshot:** Cluster assignments based on 2019 production profiles (end of panel)
# - **Aggregated:** Uses earliest available year per country across 1995-2005
# 
# **Inputs:**
# - `intermediary/NaturalResource.csv` (from Step 3)
# - `intermediary/Master.csv` (from Step 3, for the ECI evolution chart)
# 
# **Outputs:**
# - `intermediary/clusters1995.csv`, `intermediary/clusters2019.csv`, `intermediary/clustersagg.csv`
# 
# ---

# ## 0. Setup

# Fixed colour palette keyed by cluster label (label-stable, not cluster-ID-stable)
_LABEL_COLORS = {
    'Petrostates':          '#d4853b',   # orange
    'Oil Exporters':        '#4a6fa5',   # blue
    'Diversified Exporters':'#2e7d4a',   # green
    'Gold & Coal':          '#c23a3a',   # red
}

# ## 1. Load Data and Define Sample
# 
# The 54-country sample corresponds to resource-dependent developing economies identified through the filtering criteria in Steps 1-3 (total natural resource rents > 5% of GDP in 1995, non-high-income).

nr = pd.read_csv("intermediary/NaturalResource.csv")

# Countries in the analysis sample
include_list = [
    'AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
    'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
    'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
    'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
    'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
    'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE',
]

nr_sample = nr[nr["Country Code"].isin(include_list)]
print(f"NR data: {nr_sample.shape[0]:,} rows")
print(f"Countries: {nr_sample['Country Code'].nunique()}")
print(f"Years: {sorted(nr_sample['Year'].unique())}")
print(f"Resources: {nr_sample['Resource'].nunique()}")

# ## 2. Clustering Pipeline
# 
# The clustering function encapsulates the full pipeline: pivot to per-capita production values, log1p transform, PCA, K-Means. It returns the cluster assignments along with PC coordinates for plotting.
# 
# Cluster labels are assigned automatically based on the centroid position in PCA space rather than hardcoded to specific cluster numbers (since K-Means assigns arbitrary IDs that can change between runs). The labelling logic uses the sign and magnitude of PC1 and PC2 loadings: PC1 captures hydrocarbon dominance, PC2 captures mineral dominance.

def run_clustering(nr_data, year_filter=None, agg_years=None, n_clusters=4, random_state=42):
    """
    Full clustering pipeline: pivot -> per capita -> log1p -> PCA(2) -> KMeans(k).

    Parameters
    ----------
    nr_data : DataFrame
        NaturalResource data with Country, Country Code, Year, Resource,
        Production_TotalValue, Population columns.
    year_filter : int or None
        If set, restrict to a single year (e.g. 1995 or 2019).
    agg_years : list or None
        If set, restrict to these years and take earliest per country.
    n_clusters : int
        Number of K-Means clusters.
    random_state : int
        For KMeans reproducibility (PCA uses deterministic solver).

    Returns
    -------
    pca_df : DataFrame with Country, Country Code, Year, PC1, PC2, Cluster, ClusterLabels
    pca_model : fitted PCA object
    feature_cols : list of resource column names used
    """

    df = nr_data.copy()

    # ── Year selection ──
    if year_filter is not None:
        df = df[df["Year"] == year_filter]
    elif agg_years is not None:
        df = df[df["Year"].isin(agg_years)]

    # ── Pivot: one row per country, one column per resource ──
    df_pivot = df.pivot_table(
        index=["Country", "Country Code", "Year", "Population"],
        columns="Resource",
        values="Production_TotalValue",
    ).reset_index()

    resource_cols = df_pivot.columns.difference(
        ["Country", "Country Code", "Year", "Population"]
    )

    # ── Per-capita normalisation ──
    df_pivot[resource_cols] = df_pivot[resource_cols].div(
        df_pivot["Population"], axis=0
    )
    df_pivot.drop(columns="Population", inplace=True)
    df_pivot = df_pivot.fillna(0)

    # ── Take earliest year per country (for aggregated variant) ──
    df_latest = (
        df_pivot.sort_values("Year", ascending=True)
        .groupby(["Country", "Country Code"])
        .first()
        .reset_index()
    )

    feature_cols = [c for c in df_latest.columns if c not in ["Country", "Country Code", "Year"]]

    # ── log1p transform ──
    X = df_latest[feature_cols].fillna(0)
    X_log = np.log1p(X)

    # ── PCA ──
    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(X_log)

    # ── K-Means ──
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    clusters = kmeans.fit_predict(pca_components)

    # ── Build results DataFrame ──
    pca_df = pd.DataFrame({
        "Country": df_latest["Country"],
        "Country Code": df_latest["Country Code"],
        "Year": df_latest["Year"],
        "PC1": pca_components[:, 0],
        "PC2": pca_components[:, 1],
        "Cluster": clusters,
    })

    # ── Auto-label clusters from centroids ──
    # FIX (Check 11): replaced hardcoded PC1/PC2 thresholds with rank-based labeling.
    # The old thresholds (pc1 > 1.5, etc.) were calibrated on data where oil production
    # was not annualised; after the unit fix in NB3 the scale changes and hardcoded
    # absolute values no longer generalise. Rank-based assignment is invariant to scale.
    #
    # Logic:
    #   1. Cluster with highest PC1 → Oil-dominant (hydrocarbons load on PC1)
    #   2. Cluster with highest PC2 (not already labeled) → Mineral-dominant
    #   3. Remaining cluster with higher PC1 → Some Oil
    #   4. Remaining cluster with lower PC1 → No Resources
    centroids = kmeans.cluster_centers_

    pc1_rank = list(np.argsort(-centroids[:, 0]))  # highest PC1 first
    pc2_rank = list(np.argsort(-centroids[:, 1]))  # highest PC2 first

    label_map = {}
    labeled = set()

    # 1. Highest PC1 → Oil-dominant (Gulf-style petrostates, high per-capita oil)
    oil_id = pc1_rank[0]
    label_map[oil_id] = "Petrostates"
    labeled.add(oil_id)

    # 2. Highest PC2 not yet labeled → Mineral/diversified (copper, coal, gold mix)
    mineral_id = next(c for c in pc2_rank if c not in labeled)
    label_map[mineral_id] = "Diversified Exporters"
    labeled.add(mineral_id)

    # 3 & 4. Remaining two by PC1 rank
    remaining = [c for c in pc1_rank if c not in labeled]
    label_map[remaining[0]] = "Oil Exporters"
    label_map[remaining[1]] = "Gold & Coal"

    pca_df["ClusterLabels"] = pca_df["Cluster"].map(label_map)

    # ── Metrics ──
    sil = silhouette_score(pca_components, clusters)
    print(f"Silhouette score: {sil:.3f}")
    print(f"Cluster distribution:")
    for cid in sorted(pca_df["Cluster"].unique()):
        n = (pca_df["Cluster"] == cid).sum()
        print(f"  {label_map[cid]}: {n} countries")

    return pca_df, pca, feature_cols

# ## 3. Validate k with Silhouette Analysis
# 
# Before running the final clustering, we check that k=4 is a reasonable choice by computing silhouette scores for k=2 through k=8. The silhouette score measures how similar each point is to its own cluster compared to neighbouring clusters (range: -1 to 1, higher is better).

# Run validation on 1995 data
nr_1995 = nr_sample[nr_sample["Year"] == 1995].copy()

df_pivot_val = nr_1995.pivot_table(
    index=["Country", "Country Code", "Year", "Population"],
    columns="Resource",
    values="Production_TotalValue",
).reset_index()

resource_cols_val = df_pivot_val.columns.difference(
    ["Country", "Country Code", "Year", "Population"]
)
df_pivot_val[resource_cols_val] = df_pivot_val[resource_cols_val].div(
    df_pivot_val["Population"], axis=0
)
df_pivot_val = df_pivot_val.fillna(0)

feat_cols_val = [c for c in df_pivot_val.columns
                 if c not in ["Country", "Country Code", "Year", "Population"]]
X_val = np.log1p(df_pivot_val[feat_cols_val].fillna(0))

pca_val = PCA(n_components=2)
X_pca_val = pca_val.fit_transform(X_val)

# Silhouette scores for k = 2..8
k_range = range(2, 9)
sil_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca_val)
    sil_scores.append(silhouette_score(X_pca_val, labels))
    inertias.append(km.inertia_)

print("\nSilhouette scores:")
for k, s in zip(k_range, sil_scores):
    marker = " <-- selected" if k == 4 else ""
    print(f"  k={k}: {s:.3f}{marker}")

# ## 4. Run Clustering for All Three Variants
# 
# The pipeline is run three times:
# - **1995:** Single-year snapshot at the start of the panel
# - **2019:** Single-year snapshot at the end of the panel
# - **Aggregated:** Uses data from 1995, 1999, and 2005, taking the earliest available year per country. This smooths out year-specific fluctuations while still capturing structural resource profiles

results = {}

# ── 1995 snapshot ──
print("=" * 60)
print("1995 SNAPSHOT")
print("=" * 60)
pca_1995, pca_model_1995, feat_1995 = run_clustering(
    nr_sample, year_filter=1995
)
results["1995"] = pca_1995
print()

# ── 2019 snapshot ──
print("=" * 60)
print("2019 SNAPSHOT")
print("=" * 60)
pca_2019, pca_model_2019, feat_2019 = run_clustering(
    nr_sample, year_filter=2019
)
results["2019"] = pca_2019
print()

# ── Aggregated (1995, 1999, 2005) ──
print("=" * 60)
print("AGGREGATED (1995, 1999, 2005)")
print("=" * 60)
pca_agg, pca_model_agg, feat_agg = run_clustering(
    nr_sample, agg_years=[1995, 1999, 2005]
)
results["agg"] = pca_agg

# ## 5. PCA Loadings Analysis
# 
# The loadings reveal which resources drive each principal component. PC1 is expected to capture hydrocarbon abundance (oil, natural gas), while PC2 should capture mineral production (copper, gold, zinc, etc.). This interpretation is central to the cluster labelling logic.

# Use 1995 model for loadings analysis (consistent with report)
loadings = pd.DataFrame(
    pca_model_1995.components_.T,
    columns=["PC1", "PC2"],
    index=feat_1995,
)

print("PCA Explained Variance (1995):")
for i, var in enumerate(pca_model_1995.explained_variance_ratio_):
    cum = pca_model_1995.explained_variance_ratio_[:i + 1].sum()
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cum*100:.1f}%)")

print("\nTop loadings by component:")
for pc in ["PC1", "PC2"]:
    print(f"\n{pc}:")
    sorted_l = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index)
    for feat, val in sorted_l.head(8).items():
        print(f"  {feat:35s} {val:+.4f}")

# ## 6. Biplot: PCA Space with Cluster Assignments
# 
# The biplot overlays cluster assignments onto the PCA space, with arrows showing the direction and strength of each resource's contribution to the principal components. This is Figure 2 in the report.

def create_biplot(pca_df, pca_model, feature_cols, title_suffix=""):
    """Create PCA biplot with cluster colours and loading arrows."""

    loadings_plot = pca_model.components_.T * np.sqrt(pca_model.explained_variance_)
    loadings_df = pd.DataFrame(loadings_plot[:, :2], columns=["PC1", "PC2"], index=feature_cols)
    scale_factor = 2.5
    loadings_scaled = loadings_df * scale_factor

    # Top features by combined loading magnitude
    importance = loadings_df.abs().sum(axis=1)
    top_n = min(15, len(feature_cols))
    top_feats = importance.nlargest(top_n).index

    fig = px.scatter(
        pca_df, x="PC1", y="PC2",
        color="ClusterLabels",
        hover_data=["Country", "Country Code", "Year"],
        color_discrete_sequence=px.colors.qualitative.Bold,
    )

    for feat in top_feats:
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"],
            y=loadings_scaled.loc[feat, "PC2"],
            ax=0, ay=0, xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="black",
        )
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"] * 1.15,
            y=loadings_scaled.loc[feat, "PC2"] * 1.15,
            text=feat, showarrow=False,
            font=dict(size=9, color="black"),
        )

    var1 = pca_model.explained_variance_ratio_[0] * 100
    var2 = pca_model.explained_variance_ratio_[1] * 100
    fig.update_layout(
        width=1000, height=700,
        xaxis_title=f"PC1 ({var1:.1f}%)",
        yaxis_title=f"PC2 ({var2:.1f}%)",
    )
    return fig


fig_biplot = create_biplot(pca_1995, pca_model_1995, feat_1995, " — 1995")
fig_biplot.write_html(os.path.join(OUT, '03_cluster__pca_biplot_country_resource_groups.html'))
print(f'Saved: Final/charts/03_cluster__pca_biplot_country_resource_groups.html')

# ## 7. Choropleth Map with Dominance Flags
# 
# Countries producing more than 15% of global output for any single resource are flagged with a red border. This highlights major producers (e.g. Chile for copper, Saudi Arabia for oil) whose resource profiles have global significance.

def create_cluster_map(pca_df, nr_data, cluster_names_map=None, dominance_threshold=15.0):
    """Choropleth map with red borders for major global producers."""

    if cluster_names_map is None:
        cluster_names_map = dict(
            zip(pca_df["Cluster"].unique(), pca_df["ClusterLabels"].unique())
        )

    # ── Global production shares ──
    df_total = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in df_total.columns if c not in ["Country", "Country Code"]]
    for col in prod_cols:
        total = df_total[col].sum()
        if total > 0:
            df_total[f"{col}_Share"] = (df_total[col] / total) * 100

    share_cols = [c for c in df_total.columns if c.endswith("_Share")]

    # Merge shares into pca_df
    df_map = pca_df.merge(df_total[["Country Code"] + share_cols], on="Country Code", how="left")

    # Flag dominant producers (vectorised)
    df_map["Is_Dominant"] = (df_map[share_cols] >= dominance_threshold).any(axis=1)
    df_map["Dominant_Resources"] = df_map.apply(
        lambda row: [
            sc.replace("_Share", "")
            for sc in share_cols
            if row.get(sc, 0) >= dominance_threshold
        ],
        axis=1,
    )

    # ── Hover text ──
    def make_hover(row):
        lbl = row["ClusterLabels"]
        lines = [f"<b>{row['Country']}</b>", f"Cluster: {lbl}"]
        # Top resources
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        if vals:
            lines.append("<br>Top Resources:")
            for res, v in vals[:3]:
                if v > 1e9:
                    lines.append(f"  {res}: ${v/1e9:.1f}B")
                elif v > 1e6:
                    lines.append(f"  {res}: ${v/1e6:.0f}M")
                else:
                    lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df_map["hover_text"] = df_map.apply(make_hover, axis=1)

    # ── Build map ──
    fig = go.Figure()

    for cid in sorted(df_map["Cluster"].unique()):
        lbl   = cluster_names_map.get(cid, f"Cluster {cid}")
        color = _LABEL_COLORS.get(lbl, '#aaa')

        # Non-dominant countries
        sub = df_map[(df_map["Cluster"] == cid) & (~df_map["Is_Dominant"])]
        if len(sub) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub["Country Code"], z=[cid]*len(sub),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=True,
                customdata=sub["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=lbl,
                marker=dict(line=dict(color="white", width=0.6)),
            ))

        # Dominant producers — thick dark border
        sub_d = df_map[(df_map["Cluster"] == cid) & (df_map["Is_Dominant"])]
        if len(sub_d) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub_d["Country Code"], z=[cid]*len(sub_d),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=False,           # same cluster — don't duplicate legend entry
                customdata=sub_d["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{lbl} ★ major producer",
                marker=dict(line=dict(color="#111", width=2.2)),
            ))

    # Dummy legend entry explaining the black border
    fig.add_trace(go.Choropleth(
        locations=["ZZZ"], z=[0],
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        showscale=False, showlegend=True,
        name="★ >15% of global output",
        marker=dict(line=dict(color="#111", width=2.2)),
    ))

    fig.update_geos(
        projection_type="natural earth",
        showcountries=True, countrycolor="#ccc",
        showcoastlines=True, coastlinecolor="#ccc",
        showland=True, landcolor="#ffffff",
        showocean=True, oceancolor="#ffffff",
        showframe=False,
    )
    fig.update_layout(
        width=1200, height=520,
        margin=dict(l=0, r=0, t=50, b=70),
        legend=dict(
            orientation="h",
            x=0.5, y=-0.08, xanchor="center", yanchor="top",
            font=dict(size=11, family="IBM Plex Sans"),
            bgcolor="rgba(250,250,250,0.9)",
            bordercolor="#ffffff", borderwidth=1,
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="IBM Plex Sans"),
    )
    return fig


# Use 1995 for the main map (matches report Figure 3)
nr_1995_full = nr_sample[nr_sample["Year"] == 1995]
fig_map = create_cluster_map(pca_1995, nr_1995_full)
fig_map.write_html(os.path.join(OUT, '04_cluster__world_map_four_resource_profiles.html'))
print(f'Saved: Final/charts/04_cluster__world_map_four_resource_profiles.html')

# ## 8. ECI vs GDP Evolution (Rosling Chart)
# 
# This animated chart tracks how each country's Economic Complexity Index and GDP per capita evolved from 1995 to 2019. Countries are coloured by their 1995 cluster assignment (fixed throughout the animation), with bubble size proportional to production per capita. Arrows trace each country's trajectory from its 1995 starting position. This is Figure 4 in the report.

master = pd.read_csv("intermediary/Master.csv")
master = master[master["Country Code"].isin(include_list)]

# Merge cluster assignments (from aggregated clustering)
master = pd.merge(
    master,
    pca_agg[["Country Code", "Cluster", "ClusterLabels"]],
    on="Country Code",
    how="left",
)

CLUSTER_COLORS_NB4 = {
    cid: _LABEL_COLORS.get(
        pca_agg.loc[pca_agg["Cluster"] == cid, "ClusterLabels"].iloc[0], '#aaa'
    )
    for cid in sorted(pca_agg["Cluster"].unique())
}

CLUSTER_NAMES = dict(zip(pca_agg["Cluster"], pca_agg["ClusterLabels"]))

def create_rosling_chart(df, cluster_colors, cluster_names, arrow_opacity=0.5, arrow_width=2):
    """Animated ECI vs log(GDP pc) chart with trajectory arrows from 1995."""

    data = df.copy()
    data["Log GDP per capita"] = np.log(data["GDP per capita (constant prices, PPP)"])
    data["Production_Per_Capita"] = data["Total_Production_Value"] / data["Population"]

    # Fix cluster to 1995 value
    c1995 = data[data["Year"] == 1995][["Country Code", "Cluster"]].copy()
    c1995 = c1995.rename(columns={"Cluster": "Cluster_1995"})
    data = data.merge(c1995, on="Country Code", how="left")
    data = data.dropna(subset=["Cluster_1995", "Log GDP per capita",
                                "Economic Complexity Index", "Production_Per_Capita"])
    data["Cluster_1995"] = data["Cluster_1995"].astype(int)

    # Bubble size
    data["Bubble_Size"] = np.sqrt(data["Production_Per_Capita"])
    mn, mx = data["Bubble_Size"].min(), data["Bubble_Size"].max()
    data["Bubble_Size_Scaled"] = 8 + (data["Bubble_Size"] - mn) / (mx - mn) * 42

    data = data.sort_values(["Year", "Country Code"])
    years = sorted(data["Year"].unique())
    countries_list = data["Country Code"].unique()
    clusters = sorted(data["Cluster_1995"].unique())

    # Build per-country data
    cdata = {}
    for code in countries_list:
        cdf = data[data["Country Code"] == code].sort_values("Year")
        origin = cdf[cdf["Year"] == 1995]
        if len(origin) == 0:
            continue
        cdata[code] = {
            "years": cdf["Year"].values,
            "x": cdf["Log GDP per capita"].values,
            "y": cdf["Economic Complexity Index"].values,
            "x0": origin["Log GDP per capita"].values[0],
            "y0": origin["Economic Complexity Index"].values[0],
            "size": cdf["Bubble_Size_Scaled"].values,
            "name": cdf["Country Name"].iloc[0],
            "cluster": cdf["Cluster_1995"].iloc[0],
            "prod_pc": cdf["Production_Per_Capita"].values,
        }

    valid_countries = list(cdata.keys())
    first_year = years[0]

    fig = go.Figure()

    for cl in clusters:
        cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
        color = cluster_colors.get(cl, "#999999")

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            xc = cd["x"][idx[0]] if len(idx) > 0 else cd["x0"]
            yc = cd["y"][idx[0]] if len(idx) > 0 else cd["y0"]
            fig.add_trace(go.Scatter(
                x=[cd["x0"], xc], y=[cd["y0"], yc],
                mode="lines", line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity, legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            if len(idx) > 0:
                i = idx[0]
                xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
            else:
                xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
            fig.add_trace(go.Scatter(
                x=xv, y=yv, mode="markers+text",
                marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                text=[code], textposition="top center", textfont=dict(size=8, color="black"),
                name=cluster_names.get(cl, f"Cluster {cl}"),
                legendgroup=f"cl_{cl}", showlegend=(code == cc[0]),
                customdata=[[cd["name"], pv, first_year]],
                hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                              "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                              "Year: %{customdata[2]}<extra></extra>",
            ))

        for code in cc:
            cd = cdata[code]
            fig.add_trace(go.Scatter(
                x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                marker=dict(size=5, color=color, opacity=0.6, symbol="circle"),
                legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

    # Frames for animation
    frames = []
    for year in years:
        fd = []
        for cl in clusters:
            cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
            color = cluster_colors.get(cl, "#999999")
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    xc, yc = cd["x"][idx[0]], cd["y"][idx[0]]
                else:
                    mask = cd["years"] <= year
                    li = np.where(mask)[0][-1] if mask.any() else 0
                    xc, yc = cd["x"][li], cd["y"][li]
                fd.append(go.Scatter(x=[cd["x0"], xc], y=[cd["y0"], yc],
                                     mode="lines", line=dict(color=color, width=arrow_width), opacity=arrow_opacity))
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    i = idx[0]
                    xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
                else:
                    mask = cd["years"] <= year
                    if mask.any():
                        li = np.where(mask)[0][-1]
                        xv, yv, sv, pv = [cd["x"][li]], [cd["y"][li]], cd["size"][li], cd["prod_pc"][li]
                    else:
                        xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
                fd.append(go.Scatter(
                    x=xv, y=yv, mode="markers+text",
                    marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                    text=[code], textposition="top center", textfont=dict(size=8),
                    customdata=[[cd["name"], pv, year]],
                    hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                                  "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                                  "Year: %{customdata[2]}<extra></extra>",
                ))
            for code in cc:
                cd = cdata[code]
                fd.append(go.Scatter(x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                                     marker=dict(size=5, color=color, opacity=0.6, symbol="circle")))
        frames.append(go.Frame(data=fd, name=str(year)))

    fig.frames = frames

    eci_vals = data["Economic Complexity Index"]
    x_vals = data["Log GDP per capita"]
    fig.update_layout(
        xaxis=dict(range=[x_vals.min()-0.2, x_vals.max()+0.2], title="Log GDP per capita (PPP)"),
        yaxis=dict(range=[eci_vals.min()-0.5, eci_vals.max()+0.5], title="Economic Complexity Index"),
        plot_bgcolor="white", width=850, height=650,
        legend=dict(title="Resource Profile (1995)", x=1.02, y=0.99),
        updatemenus=[dict(
            type="buttons", showactive=True, x=1.0, y=-0.02,
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, dict(frame=dict(duration=500, redraw=True), transition=dict(duration=300))]),
                dict(label="Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            active=0, len=0.85, x=0.05, y=-0.12,
            currentvalue=dict(prefix="Year: ", font=dict(size=14)),
            steps=[dict(args=[[str(y)], dict(frame=dict(duration=300, redraw=True), mode="immediate")],
                        method="animate", label=str(y)) for y in years],
        )],
    )
    return fig


fig_rosling = create_rosling_chart(master, CLUSTER_COLORS_NB4, CLUSTER_NAMES)
fig_rosling.write_html(os.path.join(OUT, '05_cluster__eci_vs_gdp_animated_1995_to_2019.html'))
print(f'Saved: Final/charts/05_cluster__eci_vs_gdp_animated_1995_to_2019.html')

# ## 9. Add Hover Text and Export
# 
# The cluster CSVs include hover text with top resources for each country, matching the format used in the original analysis files.

def add_hover_text(pca_df, nr_data):
    """Add hover text with top resources to cluster DataFrame."""
    # Get total production by country and resource
    totals = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in totals.columns if c not in ["Country", "Country Code"]]

    df = pca_df.merge(totals, on="Country Code", how="left", suffixes=("", "_nr"))

    def make_ht(row):
        lines = [f"<b>{row['Country']}</b>",
                 f"Cluster: {row['Cluster']}"]
        lines.append("<br>Top Resources:")
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        for res, v in vals[:3]:
            lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df["hover_text"] = df.apply(make_ht, axis=1)
    return df[["Country", "Country Code", "Year", "PC1", "PC2",
               "Cluster", "ClusterLabels", "hover_text"]]


# Export each variant
for label, df, nr_subset in [
    ("1995", results["1995"], nr_sample[nr_sample["Year"] == 1995]),
    ("2019", results["2019"], nr_sample[nr_sample["Year"] == 2019]),
    ("agg",  results["agg"],  nr_sample[nr_sample["Year"].isin([1995, 1999, 2005])]),
]:
    out = add_hover_text(df, nr_subset)
    out_path = f"intermediary/clusters{label}.csv"
    out.to_csv(out_path, index=False)
    print(f"Saved {out_path}: {len(out)} countries")

print("\nDone. All cluster CSVs exported.")

# ── NB4 SUMMARY ──
print("=" * 70)
print("NB4: CLUSTERING SUMMARY")
print("=" * 70)

print(f"\nPCA variance explained (1995): PC1={pca_model_1995.explained_variance_ratio_[0]*100:.1f}%, PC2={pca_model_1995.explained_variance_ratio_[1]*100:.1f}%")

for label, df in results.items():
    print(f"\n--- {label} ({len(df)} countries) ---")
    for cid in sorted(df['Cluster'].unique()):
        sub = df[df['Cluster'] == cid]
        lbl = sub['ClusterLabels'].iloc[0]
        codes = ', '.join(sorted(sub['Country Code'].tolist()))
        print(f"  {lbl} (n={len(sub)}): {codes}")

print(f"\nSaved:")
for label in results:
    print(f"  intermediary/clusters{label}.csv")

## Section 2 — ML Training (NB5): Feature Engineering & Model Fitting

This section runs the full supervised learning pipeline:
- Feature engineering (log transforms, interaction terms, lag)
- Temporal train/test split (train: 1995–2014, test: 2015–2019)
- Model training (LASSO, Ridge, Elastic Net, Random Forest)
- Diagnostics (VIF, performance, prediction intervals)
- Forecast 2020–2030
- All intermediate CSVs saved to `Final/NB5/`

**Note:** This section saves CSV outputs used by Section 3 to render charts 07–13.

In [ ]:
OUT_CHARTS = 'Final/charts'

# # Supervised Learning: Drivers of Economic Complexity
# 
# **Moody's Ratings Capstone — Industrial Upgrading in Emerging Markets**
# **NB5 of 6 · Pipeline Step: Modelling**
# 
# Unified notebook combining the interpretation pipeline (`5_ML_FINAL`) and the
# validation/forecasting pipeline (`5_ML_TEST`).
# 
# | Section | Content |
# |---|---|
# | 0–2 | Setup, feature engineering, temporal train/test split |
# | 3–4 | Model training (LASSO, Ridge, Elastic Net, RF, XGBoost) + diagnostics |
# | 5 | Static charts: VIF, coefficients, model agreement, RF importance, OOS R², SHAP, PI |
# | 6 | Interactive Plotly charts |
# | 7 | Coefficient summary table |
# | 8 | Forecast 2020–2030 with country rankings |
# | 9 | Summary |

# ## 0. Setup and Data Loading

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
# TimeSeriesSplit replaced by PanelTemporalCV (defined in Cell 6)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — run: pip install xgboost")

try:
    import shap
    HAS_SHAP = True
    print("SHAP available")
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed — run: pip install shap")

# ── Output directory ─────────────────────────────────────────────────────────
OUT = os.path.join('Final', 'NB5')
os.makedirs(OUT, exist_ok=True)

# ── Shared style (IBM Plex Sans, #fafafa, navy/red palette) ─────────────────
STYLE = {
    'font_family':       'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size':         11,
    'axis_title_size':   13,
    'legend_size':       11,
    'annotation_size':   11,
    'title_color':       '#1a2744',
    'template':          'plotly_white',
    'plot_bg':           '#fafafa',
    'paper_bg':          '#fafafa',
    'chart_height':      550,
    'chart_height_small':420,
    'chart_height_tall': 700,
    'margin':            dict(l=60,  r=40,  t=10, b=50),
    'margin_bar':        dict(l=160, r=130, t=10, b=50),
    'grid_color':        '#e5e7eb',
    'grid_width':        0.5,
    'zero_line_color':   '#c9cfd6',
}
PALETTE = {
    'blue':        '#4a6fa5',
    'red':         '#c23a3a',
    'green':       '#2e7d4a',
    'orange':      '#d4853b',
    'light_blue':  '#7a9dc4',
    'light_red':   '#d46b6b',
    'light_green': '#5aa87a',
    'dark':        '#3d4f5f',
    'grey':        '#999999',
    'gold':        '#e6b980',
}
WRITE_CONFIG = {'displayModeBar': False, 'responsive': True}

plt.rcParams.update({
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
})


def base_layout(**kwargs) -> dict:
    layout = dict(
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        font=dict(family=STYLE['font_family'], size=STYLE['tick_size'],
                  color=STYLE['title_color']),
        margin=STYLE['margin'],
        height=STYLE['chart_height'],
    )
    layout.update(kwargs)
    return layout


def save_chart(fig, path_no_ext: str, width: int = 1100, height: int = 700):
    """No-op: chart output is handled by viz_updates.py / Final/charts/."""
    pass


# ── Load data ─────────────────────────────────────────────────────────────────
master = pd.read_csv('intermediary/Master.csv')
print(f"Master: {len(master):,} obs, {master['Country Code'].nunique()} countries")

include_list = [
    'AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
    'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
    'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
    'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
    'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
    'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE',
]

df = master[
    (master['Year'] >= 1995) &
    (master['Year'] <= 2019) &
    (master['Country Code'].isin(include_list))
].copy()
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)
print(f"Sample: {df['Country Code'].nunique()} countries, {len(df):,} obs")

clusters = pd.read_csv('intermediary/clustersagg.csv')
cluster_map = clusters[['Country Code', 'ClusterLabels']].drop_duplicates('Country Code')
df = df.merge(cluster_map, on='Country Code', how='left')
print(f"Cluster coverage: {df['ClusterLabels'].notna().sum()} / {len(df)} rows")

# ── Per-capita production value (align with NB6) ─────────────────────────────
# NB6 uses Total_Production_Value_Per_Capita throughout. NB5 now does the same
# so coefficients / importances are on a comparable intensive-margin measure.
df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population']
)

# ── Configuration toggles ─────────────────────────────────────────────────────
# Set these before running the notebook to enable optional analyses.
RUN_NO_LAG_ROBUSTNESS = False  # True → also fit models without L1_ECI (Cell 4 diagnostic)
RUN_XGB_GRID_SEARCH   = False  # True → search XGBoost hyperparameters instead of using defaults (slow)


# ## 1. Feature Engineering
#
# Interaction terms use raw (not centred) values, consistent with the original specification.

# ── ECI targets ───────────────────────────────────────────────────────────────
df['L1_ECI']    = df.groupby('Country Code')['Economic Complexity Index'].shift(1)
df['ECI_delta'] = df['Economic Complexity Index'] - df['L1_ECI']
df = df.dropna(subset=['L1_ECI', 'Economic Complexity Index', 'ECI_delta'])

# ── Log transforms — applied BEFORE interaction computation ───────────────────
# Interactions must be built on the same scale as the constituent main effects.
# Previously interactions used raw values while main effects were subsequently
# log-transformed, causing a scale mismatch that inflated VIF (the VIF chart
# was diagnosing a problem introduced by the coding choice itself).
log_cols = [
    'Human capital index',
    'Total_Production_Value_Per_Capita',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Government revenue',
    'Use of IMF credit (DOD, current US$)',
]
df[log_cols] = np.log1p(df[log_cols]).replace([np.inf, -np.inf], np.nan)

# ── Interaction terms (log-transformed, mean-centred) ─────────────────────────
# Grand-mean centring reduces multicollinearity between the interaction term and
# its constituents. After centring, main-effect coefficients are interpreted at
# the sample mean of the interacted variable — consistent with NB6's approach.
# Means stored as module-level vars for reuse in the forecast loop (Cell 8).
_hci_mean  = df['Human capital index'].mean()
_prod_mean = df['Total_Production_Value_Per_Capita'].mean()
_rol_mean  = df['Rule of law index'].mean()

df['HCI_x_ProductionValue']       = (df['Human capital index'] - _hci_mean)  * \
                                     (df['Total_Production_Value_Per_Capita']  - _prod_mean)
df['RuleOfLaw_x_ProductionValue'] = (df['Rule of law index']    - _rol_mean)  * \
                                     (df['Total_Production_Value_Per_Capita']  - _prod_mean)

# ── Feature lists ─────────────────────────────────────────────────────────────
base_features = [
    'Total_Production_Value_Per_Capita',
    'Human capital index',
    'Rule of law index',
    'Political stability — estimate',
    'Trade (% of GDP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP',
    'Domestic credit to private sector (% of GDP)',
    'Landlocked',
    'Urban population (% of total population)',
    'Government revenue',
    'Capital depreciation rate',
    'Use of IMF credit (DOD, current US$)',
    'Real interest rate (%)',
    'Inflation, consumer prices (annual %)',
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'L1_ECI',
]
interaction_features = ['HCI_x_ProductionValue', 'RuleOfLaw_x_ProductionValue']
all_features         = base_features + interaction_features
df = df.dropna(subset=all_features)

# ── Short name mapping ────────────────────────────────────────────────────────
name_mapping = {
    'Human capital index':                                                   'Human Capital',
    'Rule of law index':                                                     'Rule of Law',
    'Political stability — estimate':                                        'Political Stability',
    'Trade (% of GDP)':                                                      'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP':   'Capital Formation',
    'Share of investment in GDP':                                            'Investment Share',
    'Domestic credit to private sector (% of GDP)':                         'Domestic Credit',
    'Landlocked':                                                            'Landlocked',
    'Urban population (% of total population)':                             'Urban Population',
    'Government revenue':                                                    'Gov Revenue',
    'Capital depreciation rate':                                             'Depreciation',
    'Use of IMF credit (DOD, current US$)':                                 'IMF Credit',
    'Real interest rate (%)':                                                'Interest Rate',
    'Inflation, consumer prices (annual %)':                                 'Inflation',
    'Access to electricity (% of population)':                              'Electricity',
    'Adjusted savings: gross savings (% of GNI)':                           'Savings',
    'Total_Production_Value_Per_Capita':                                     'Prod Value p.c.',
    'HCI_x_ProductionValue':                                                 'HC × Production',
    'RuleOfLaw_x_ProductionValue':                                           'RuleLaw × Production',
    'L1_ECI':                                                                'Lagged ECI',
}
def shorten(f): return name_mapping.get(f, f[:22])
short_names = [shorten(f) for f in all_features]

EXCLUDE = 'L1_ECI'  # excluded from importance charts (trivial predictor)

print(f"Features: {len(all_features)} ({len(base_features)} base + {len(interaction_features)} interactions)")
print(f"After transforms + dropna: {len(df):,} obs, {df['Country Code'].nunique()} countries")


# ## 2. Temporal Train / Test Split
# 
# Train: 1995–2014 (≈ 20 years). Test: 2015–2019 (5 years, fully held out).
# **The scaler is fit on the training set only** to prevent leakage of future statistics
# into the feature matrix used for cross-validation and evaluation.
# 
# Cross-validation uses `PanelTemporalCV`, an expanding-window scheme that splits
# by calendar year with a 1-year gap. This avoids the row-index problem that
# `TimeSeriesSplit` causes on panel data sorted by (country, year).
# 

class PanelTemporalCV:
    """
    Expanding-window cross-validation for panel data.

    Each fold trains on all observations with Year <= cutoff, skips a gap,
    and validates on observations with Year > cutoff + gap.

    Parameters
    ----------
    years : array-like
        The Year column corresponding to the rows of X.
    n_splits : int
        Number of CV folds.
    gap : int
        Years to skip between training and validation (default 1).
    min_train_years : int
        Minimum number of distinct years in the first training fold (default 8).
    """

    def __init__(self, years, n_splits=5, gap=1, min_train_years=8):
        self.years = np.asarray(years)
        self.n_splits = n_splits
        self.gap = gap
        self.min_train_years = min_train_years

        unique_years = np.sort(np.unique(self.years))
        min_year = unique_years[0]
        max_year = unique_years[-1]

        earliest_cutoff = min_year + self.min_train_years - 1
        latest_cutoff   = max_year - self.gap - 1

        if earliest_cutoff > latest_cutoff:
            raise ValueError(
                f"Cannot create {n_splits} folds: year range "
                f"[{min_year}, {max_year}] too narrow for gap={gap}, "
                f"min_train_years={min_train_years}."
            )

        self.cutoffs = np.unique(
            np.linspace(earliest_cutoff, latest_cutoff, n_splits).astype(int)
        )
        self.n_splits = len(self.cutoffs)

    def split(self, X=None, y=None, groups=None):
        for cutoff in self.cutoffs:
            train_mask = self.years <= cutoff
            val_mask   = self.years > cutoff + self.gap
            train_idx  = np.where(train_mask)[0]
            val_idx    = np.where(val_mask)[0]
            if len(train_idx) > 0 and len(val_idx) > 0:
                yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def __iter__(self):
        return self.split()


# ── Train / test split ────────────────────────────────────────────────────────
TRAIN_END  = 2014
TEST_START = 2015

train_df = df[df['Year'] <= TRAIN_END].copy()
test_df  = df[df['Year'] >= TEST_START].copy()

print(f"Train: {len(train_df):,} obs | {int(train_df['Year'].min())}–{TRAIN_END} | {train_df['Country Code'].nunique()} countries")
print(f"Test : {len(test_df):,}  obs | {TEST_START}–{int(test_df['Year'].max())}  | {test_df['Country Code'].nunique()} countries")

# ── Two targets ───────────────────────────────────────────────────────────────
y_train_level = train_df['Economic Complexity Index'].values
y_test_level  = test_df['Economic Complexity Index'].values
y_train_delta = train_df['ECI_delta'].values
y_test_delta  = test_df['ECI_delta'].values

# ── Scale: fit on train only, transform both ─────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[all_features].values)
X_test  = scaler.transform(test_df[all_features].values)  # transform only

# ── Temporal CV: expanding window, 1-year gap ────────────────────────────────
train_years = train_df['Year'].values
tscv = PanelTemporalCV(train_years, n_splits=5, gap=1, min_train_years=8)

print(f"\nTemporal CV folds ({tscv.n_splits} splits, gap=1 year):")
for i, (tr_idx, val_idx) in enumerate(tscv.split()):
    tr_yrs = np.unique(train_years[tr_idx])
    val_yrs = np.unique(train_years[val_idx])
    print(f"  Fold {i+1}: train {tr_yrs.min()}–{tr_yrs.max()} "
          f"({len(tr_idx):,} obs) → val {val_yrs.min()}–{val_yrs.max()} "
          f"({len(val_idx):,} obs)")

print("\nScaler fit on train only — test set transformed with train statistics.")


# ## 3. Model Training
# 
# Five models (+ optional XGBoost) trained on the training set.
# `fit_all_models` is called twice: once for the ECI level target, once for ΔECI.

def fit_all_models(X_tr, y_tr, tscv, label='', years=None):
    """
    Fit LASSO, Ridge, Elastic Net, RF, optionally XGBoost.

    Parameters
    ----------
    years : array-like or None
        Year values for each row in X_tr. Used to create a temporal
        validation split for XGBoost early stopping. If None, falls back
        to the last 15% of rows.
    """
    print(f"\nFitting models — target: {label}")
    lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_tr, y_tr)
    ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_tr, y_tr)
    elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42,
                           max_iter=10000).fit(X_tr, y_tr)
    rf      = RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=10,
        random_state=42, n_jobs=-1, oob_score=True,
    ).fit(X_tr, y_tr)

    out = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}

    if HAS_XGB:
        # ── Temporal validation split for early stopping ─────────────────
        if years is not None:
            years_arr  = np.asarray(years)
            unique_yrs = np.sort(np.unique(years_arr))
            val_cutoff = unique_yrs[-4]  # last 3 years of training window
            xgb_train_mask = years_arr <= val_cutoff
            xgb_val_mask   = years_arr >  val_cutoff
            print(f"  XGBoost early-stop split: train <={val_cutoff} "
                  f"({xgb_train_mask.sum():,}), val >{val_cutoff} "
                  f"({xgb_val_mask.sum():,})")
        else:
            n_val = max(10, int(0.15 * len(X_tr)))
            xgb_train_mask = np.ones(len(X_tr), dtype=bool)
            xgb_train_mask[-n_val:] = False
            xgb_val_mask = ~xgb_train_mask

        if RUN_XGB_GRID_SEARCH:
            # ── Small grid search (2×2×2 = 8 fits) ──────────────────────
            # Covers the two most impactful XGBoost hyperparameters for
            # small panels: tree depth (bias-variance) and column sampling.
            from itertools import product as _iproduct
            _grid = {
                'max_depth':        [2, 3],
                'subsample':        [0.6, 0.8],
                'colsample_bytree': [0.7, 0.9],
            }
            best_score, best_params = np.inf, {}
            for _md, _ss, _cbt in _iproduct(
                _grid['max_depth'], _grid['subsample'], _grid['colsample_bytree']
            ):
                _m = xgb.XGBRegressor(
                    n_estimators=300, max_depth=_md, learning_rate=0.01,
                    subsample=_ss, colsample_bytree=_cbt, min_child_weight=10,
                    random_state=42, n_jobs=-1,
                    early_stopping_rounds=20, eval_metric='rmse',
                )
                _m.fit(
                    X_tr[xgb_train_mask], y_tr[xgb_train_mask],
                    eval_set=[(X_tr[xgb_val_mask], y_tr[xgb_val_mask])],
                    verbose=False,
                )
                if _m.best_score < best_score:
                    best_score  = _m.best_score
                    best_params = {'max_depth': _md, 'subsample': _ss, 'colsample_bytree': _cbt}
            print(f"  XGBoost grid search best: {best_params}  val RMSE={best_score:.4f}")
            xgb_model = xgb.XGBRegressor(
                n_estimators=500, learning_rate=0.01, min_child_weight=10,
                random_state=42, n_jobs=-1,
                early_stopping_rounds=30, eval_metric='rmse',
                **best_params,
            )
        else:
            # ── Default conservative hyperparameters ──────────────────────
            # max_depth=2, min_child_weight=10 limit complexity on a small
            # panel. colsample_bytree=0.87 was hand-tuned; set
            # RUN_XGB_GRID_SEARCH=True (Cell 2) to search over alternatives.
            xgb_model = xgb.XGBRegressor(
                n_estimators=500, max_depth=2, learning_rate=0.01,
                subsample=0.7, colsample_bytree=0.87, min_child_weight=10,
                random_state=42, n_jobs=-1,
                early_stopping_rounds=30, eval_metric='rmse',
            )

        xgb_model.fit(
            X_tr[xgb_train_mask], y_tr[xgb_train_mask],
            eval_set=[(X_tr[xgb_val_mask], y_tr[xgb_val_mask])],
            verbose=False,
        )
        out['XGBoost'] = xgb_model
        print(f"  XGBoost best iteration: {xgb_model.best_iteration}")

    print(f"  LASSO α={lasso.alpha_:.4f}  Ridge α={ridge.alpha_:.4f}  "
          f"EN α={elastic.alpha_:.4f}  RF OOB={rf.oob_score_:.3f}")
    return out


models_level = fit_all_models(X_train, y_train_level, tscv, label='ECI',
                              years=train_years)
models_delta = fit_all_models(X_train, y_train_delta, tscv, label='ΔECI',
                              years=train_years)


# ## 4. Diagnostics
# 
# VIF computed on `X_train` (scaled training set only).
# Train and test R² reported side-by-side to diagnose overfitting.

# ── VIF on training set ───────────────────────────────────────────────────────
vif_data = pd.DataFrame({
    'Feature': all_features,
    'VIF':     [variance_inflation_factor(X_train, i)
                for i in range(X_train.shape[1])],
}).sort_values('VIF', ascending=False).reset_index(drop=True)

high_vif = vif_data[vif_data['VIF'] > 10]
print(f"High-VIF features (>10):")
print(high_vif.to_string(index=False) if len(high_vif) else "  None")

# ── Train vs test performance ─────────────────────────────────────────────────
def eval_models(models, X_tr, y_tr, X_te, y_te):
    rows = []
    for name, model in models.items():
        pred_tr = model.predict(X_tr)
        pred_te = model.predict(X_te)
        rows.append({
            'Model':       name,
            'Train R²':   round(r2_score(y_tr, pred_tr), 4),
            'Test R²':    round(r2_score(y_te, pred_te), 4),
            'Train RMSE': round(np.sqrt(mean_squared_error(y_tr, pred_tr)), 4),
            'Test RMSE':  round(np.sqrt(mean_squared_error(y_te, pred_te)), 4),
            'Train MAE':  round(mean_absolute_error(y_tr, pred_tr), 4),
            'Test MAE':   round(mean_absolute_error(y_te, pred_te), 4),
            'Overfit Gap': round(r2_score(y_tr, pred_tr) - r2_score(y_te, pred_te), 4),
        })
    return pd.DataFrame(rows).sort_values('Test R²', ascending=False).reset_index(drop=True)

perf_level = eval_models(models_level, X_train, y_train_level, X_test, y_test_level)
perf_delta = eval_models(models_delta, X_train, y_train_delta, X_test, y_test_delta)

perf_level.to_csv(os.path.join(OUT, 'model_performance_level.csv'), index=False)
perf_delta.to_csv(os.path.join(OUT, 'model_performance_delta.csv'), index=False)

print("\n── Target: ECI ──────────────────────────────────────────────────────────")
print(perf_level.to_string(index=False))
print("\n── Target: ΔECI ─────────────────────────────────────────────────────────")
print(perf_delta.to_string(index=False))

# ── Normalised importance & signed coefficients ───────────────────────────────
def minmax(arr):
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo) if hi > lo else arr

all_importance = pd.DataFrame({'Feature': all_features})
for name, model in models_level.items():
    if name == 'Random Forest':
        all_importance[name] = minmax(model.feature_importances_)
    elif hasattr(model, 'coef_'):
        all_importance[name] = minmax(np.abs(model.coef_))

lin_cols = [n for n in ['LASSO', 'Ridge', 'Elastic Net'] if n in all_importance]
all_importance['Linear_Avg']  = all_importance[lin_cols].mean(axis=1)
if 'Random Forest' in all_importance:
    all_importance['Overall_Avg'] = all_importance[['Linear_Avg', 'Random Forest']].mean(axis=1)
else:
    all_importance['Overall_Avg'] = all_importance['Linear_Avg']
all_importance = all_importance.sort_values('Overall_Avg', ascending=False).reset_index(drop=True)
all_importance.to_csv(os.path.join(OUT, 'all_importance.csv'), index=False)

lasso   = models_level['LASSO']
ridge   = models_level['Ridge']
elastic = models_level['Elastic Net']
rf      = models_level['Random Forest']
lin_models = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic}

lasso_sel = int(np.sum(lasso.coef_ != 0))
en_sel    = int(np.sum(elastic.coef_ != 0))
top5 = (all_importance[all_importance['Feature'] != EXCLUDE]
        .head(5)['Feature'].apply(shorten).tolist())
print(f"\n  LASSO selected {lasso_sel}/{len(all_features)} features")
print(f"  Elastic Net selected {en_sel}/{len(all_features)} features")
print(f"  Top 5 by Elastic Net importance: {top5}")

# ── SHAP (if available) ───────────────────────────────────────────────────────
shap_results = {}
if HAS_SHAP:
    print("\nComputing SHAP — Random Forest (test set)...")
    rf_explainer = shap.TreeExplainer(models_level['Random Forest'])
    shap_results['Random Forest'] = rf_explainer.shap_values(X_test)
    if HAS_XGB and 'XGBoost' in models_level:
        print("Computing SHAP — XGBoost (native booster)...")
        dmatrix = xgb.DMatrix(X_test, feature_names=short_names)
        xgb_contribs = models_level['XGBoost'].get_booster().predict(dmatrix, pred_contribs=True)
        shap_results['XGBoost'] = xgb_contribs[:, :-1]
    for mname, sv in shap_results.items():
        mean_abs = np.abs(sv).mean(axis=0)
        feat_idx = [i for i, f in enumerate(all_features) if f != EXCLUDE]
        top3_idx = np.argsort(mean_abs[feat_idx])[::-1][:3]
        print(f"  SHAP top 3 ({mname}): {[short_names[feat_idx[i]] for i in top3_idx]}")

# ── Prediction intervals (quantile GB) ────────────────────────────────────────
gb_params = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
                 subsample=0.8, min_samples_leaf=10, random_state=42)
gb_q10 = GradientBoostingRegressor(loss='quantile', alpha=0.10, **gb_params).fit(X_train, y_train_level)
gb_q50 = GradientBoostingRegressor(loss='quantile', alpha=0.50, **gb_params).fit(X_train, y_train_level)
gb_q90 = GradientBoostingRegressor(loss='quantile', alpha=0.90, **gb_params).fit(X_train, y_train_level)

pred_q10 = gb_q10.predict(X_test)
pred_q50 = gb_q50.predict(X_test)
pred_q90 = gb_q90.predict(X_test)

coverage  = np.mean((y_test_level >= pred_q10) & (y_test_level <= pred_q90))
avg_width = np.mean(pred_q90 - pred_q10)
print(f"\n80% prediction interval — coverage: {coverage:.1%}  avg width: {avg_width:.4f}")

interval_df = test_df[['Country Code', 'Country Name', 'Year']].copy().reset_index(drop=True)
interval_df['Actual']  = y_test_level
interval_df['Q10']     = pred_q10
interval_df['Q50']     = pred_q50
interval_df['Q90']     = pred_q90
interval_df['In_Band'] = (interval_df['Actual'] >= interval_df['Q10']) & \
                          (interval_df['Actual'] <= interval_df['Q90'])

# ── L1_ECI robustness check (toggle in Cell 2) ───────────────────────────────
# Motivation: L1_ECI dominates every model (~0.97 autocorrelation). Coefficients
# on substantive variables are estimated conditional on persistence being
# controlled for. Running without L1_ECI shows what matters for ECI levels
# in the absence of that control — complementary to the delta-target models.
if RUN_NO_LAG_ROBUSTNESS:
    _feats_nl = [f for f in all_features if f != 'L1_ECI']
    _scaler_nl = StandardScaler()
    _X_train_nl = _scaler_nl.fit_transform(train_df[_feats_nl].values)
    _X_test_nl  = _scaler_nl.transform(test_df[_feats_nl].values)
    _tscv_nl = PanelTemporalCV(train_years, n_splits=5, gap=1, min_train_years=8)
    _models_nl = fit_all_models(_X_train_nl, y_train_level, _tscv_nl,
                                label='ECI (no L1_ECI)', years=train_years)
    _perf_nl = eval_models(_models_nl, _X_train_nl, y_train_level,
                           _X_test_nl, y_test_level)
    print('\n── Robustness: Models without L1_ECI ──────────────────────────────────────')
    print(_perf_nl[['Model', 'Train R²', 'Test R²', 'Overfit Gap']].to_string(index=False))
    _comp = (
        perf_level[['Model', 'Test R²']]
        .merge(_perf_nl[['Model', 'Test R²']], on='Model', suffixes=(' (with L1)', ' (no L1)'))
    )
    _comp['R² drop'] = _comp['Test R² (with L1)'] - _comp['Test R² (no L1)']
    print('\n── Test R² comparison (with vs without L1_ECI) ────────────────────────────')
    print(_comp.to_string(index=False))


# ## 5. Visualisations — Static (Matplotlib)
# 
# Eight charts:
# 1. Train vs Test R² (overfitting diagnostic)
# 2. Predicted vs Actual — test set
# 3. SHAP feature importance (if available)
# 4. 80% prediction intervals — test set
# 5. VIF multicollinearity diagnostics
# 6. 3-panel signed coefficient comparison (LASSO, Ridge, Elastic Net)
# 7. Model agreement — dot-and-range plot
# 8. Random Forest feature importance

# ── [1/8] Train vs Test R² ────────────────────────────────────────────────────
print("[1/8] Train vs Test R² comparison...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, perf, title in zip(axes,
                            [perf_level, perf_delta],
                            ['Target: ECI', 'Target: ΔECI']):
    model_names = perf['Model'].tolist()
    x = np.arange(len(model_names))
    w = 0.35
    ax.bar(x - w/2, perf['Train R²'], w, label='Train R²',
           color=PALETTE['blue'], alpha=0.85, edgecolor='black', linewidth=1)
    ax.bar(x + w/2, perf['Test R²'],  w, label='Test R²',
           color=PALETTE['red'],  alpha=0.85, edgecolor='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=11, fontweight='bold')
    ax.set_ylabel('R²', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    for i, (tr, te) in enumerate(zip(perf['Train R²'], perf['Test R²'])):
        ax.text(i, max(tr, te) + 0.015, f'Δ={tr-te:+.3f}',
                ha='center', fontsize=8.5, color='#555',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

plt.suptitle('Train vs Test R² — Overfitting Diagnostic', fontsize=15, fontweight='bold', y=1.02)
plt.figtext(0.5, -0.02, 'Train: 1995–2014  |  Test: 2015–2019  |  Δ = train R² − test R²',
            ha='center', fontsize=10, color='#444')
plt.tight_layout()
plt.close("all")


# ── CHART 1b: Train vs Test RMSE ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, perf, title in zip(axes,
                            [perf_level, perf_delta],
                            ['Target: ECI', 'Target: ΔECI']):
    model_names = perf['Model'].tolist()
    x = np.arange(len(model_names))
    w = 0.35
    ax.bar(x - w/2, perf['Train RMSE'], w, label='Train RMSE',
           color=PALETTE['blue'], alpha=0.85, edgecolor='black', linewidth=1)
    ax.bar(x + w/2, perf['Test RMSE'],  w, label='Test RMSE',
           color=PALETTE['red'],  alpha=0.85, edgecolor='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=11, fontweight='bold')
    ax.set_ylabel('RMSE', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    for i, (tr, te) in enumerate(zip(perf['Train RMSE'], perf['Test RMSE'])):
        ax.text(i, max(tr, te) + 0.002, f'Δ={te-tr:+.4f}',
                ha='center', fontsize=8.5, color='#555',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

plt.suptitle('Train vs Test RMSE — Overfitting Diagnostic', fontsize=15, fontweight='bold', y=1.02)
plt.figtext(0.5, -0.02, 'Train: 1995–2014  |  Test: 2015–2019  |  Δ = test RMSE − train RMSE',
            ha='center', fontsize=10, color='#444')
plt.tight_layout()
plt.close("all")

# ── [2/8] Predicted vs Actual ─────────────────────────────────────────────────
print("\n[2/8] Predicted vs Actual (test set)...")

best_level = perf_level.iloc[0]['Model']
best_delta = perf_delta.iloc[0]['Model']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, actual, pred_vals, name, target in zip(
    axes,
    [y_test_level,  y_test_delta],
    [models_level[best_level].predict(X_test), models_delta[best_delta].predict(X_test)],
    [best_level,    best_delta],
    ['ECI',         'ΔECI'],
):
    r2_val   = r2_score(actual, pred_vals)
    rmse_val = np.sqrt(mean_squared_error(actual, pred_vals))
    ax.scatter(actual, pred_vals, alpha=0.45, s=30, color=PALETTE['blue'], edgecolor='none')
    lims = [min(actual.min(), pred_vals.min()) - 0.05,
            max(actual.max(), pred_vals.max()) + 0.05]
    ax.plot(lims, lims, '--', color=PALETTE['red'], linewidth=1.5, alpha=0.8, label='45° line')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(f'Actual {target}', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'Predicted {target}', fontsize=12, fontweight='bold')
    ax.set_title(f'{name} — {target}  |  Test R²={r2_val:.3f}  RMSE={rmse_val:.4f}',
                 fontsize=12, fontweight='bold', pad=10)
    ax.legend(fontsize=10); ax.grid(alpha=0.2, linestyle='--')

plt.suptitle('Predicted vs Actual — Test Set (2015–2019)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.close("all")

# Save test predictions for improved chart
_pred_level = models_level[best_level].predict(X_test)
_pred_delta = models_delta[best_delta].predict(X_test)
_pred_df = test_df[['Country Code', 'Country Name', 'Year']].copy().reset_index(drop=True)
_pred_df['Actual_ECI']        = y_test_level
_pred_df['Predicted_ECI']     = _pred_level
_pred_df['Actual_Delta']      = y_test_delta
_pred_df['Predicted_Delta']   = _pred_delta
_pred_df['Best_Level_Model']  = best_level
_pred_df['Best_Delta_Model']  = best_delta
_pred_df.to_csv(os.path.join(OUT, 'test_predictions.csv'), index=False)


# ── [3/8] SHAP ────────────────────────────────────────────────────────────────
print("\n[3/8] SHAP importance...")
if HAS_SHAP and shap_results:
    n_panels = len(shap_results)
    fig, axes = plt.subplots(1, n_panels, figsize=(9 * n_panels, 9))
    if n_panels == 1: axes = [axes]
    for ax, (mname, sv) in zip(axes, shap_results.items()):
        mean_abs = np.abs(sv).mean(axis=0)
        feat_idx = [i for i, f in enumerate(all_features) if f != EXCLUDE]
        top15    = np.argsort(mean_abs[feat_idx])[-15:]
        vals_top = mean_abs[feat_idx][top15]
        names_top = [short_names[feat_idx[i]] for i in top15]
        cmap = plt.cm.YlOrRd
        colors = [cmap(0.3 + 0.65 * v / vals_top.max()) for v in vals_top]
        ax.barh(np.arange(len(top15)), vals_top, color=colors, edgecolor='black', linewidth=1.1, alpha=0.92)
        ax.set_yticks(np.arange(len(top15))); ax.set_yticklabels(names_top, fontsize=11, fontweight='bold')
        ax.set_xlabel('Mean |SHAP| (ECI units)', fontsize=11, fontweight='bold')
        ax.set_title(f'SHAP — {mname} (test set)', fontsize=12, fontweight='bold', pad=12)
        ax.grid(axis='x', alpha=0.25, linestyle='--')
    plt.suptitle('SHAP Feature Importance — Test Set', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.close("all")
    else:
    print("  Skipped — SHAP not available")


# ── [4/8] Prediction intervals ────────────────────────────────────────────────
print("\n[4/8] Prediction intervals...")
plot_df = interval_df.sort_values('Actual').reset_index(drop=True)
x_idx   = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(x_idx, plot_df['Q10'], plot_df['Q90'],
                alpha=0.25, color=PALETTE['blue'], label='80% prediction band')
ax.plot(x_idx, plot_df['Q50'], color=PALETTE['blue'], linewidth=2,
        label='Median (Q50)', alpha=0.9)
ax.scatter(x_idx[plot_df['In_Band']],  plot_df.loc[plot_df['In_Band'],  'Actual'],
           color=PALETTE['green'], s=25, alpha=0.8, zorder=3, label='Actual (in band)')
ax.scatter(x_idx[~plot_df['In_Band']], plot_df.loc[~plot_df['In_Band'], 'Actual'],
           color=PALETTE['red'],   s=35, alpha=0.9, zorder=4, marker='D', label='Actual (out of band)')
ax.set_xlabel('Observations (sorted by actual ECI)', fontsize=12, fontweight='bold')
ax.set_ylabel('ECI', fontsize=12, fontweight='bold')
ax.set_title(f'80% Prediction Intervals — Test Set (2015–2019)\n'
             f'Coverage: {coverage:.1%}  |  Avg width: {avg_width:.4f}',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper left', framealpha=0.95)
ax.grid(alpha=0.2, linestyle='--')
plt.tight_layout()
plt.close("all")


# ── [5/8] VIF ─────────────────────────────────────────────────────────────────
print("\n[5/8] VIF chart...")
vif_plot = vif_data.copy()
vif_plot['Short'] = vif_plot['Feature'].apply(shorten)
vif_plot = vif_plot.sort_values('VIF', ascending=True).reset_index(drop=True)
bar_colors = [PALETTE['red'] if v > 10 else PALETTE['blue'] for v in vif_plot['VIF']]

fig, ax = plt.subplots(figsize=(11, 8))
y_pos = np.arange(len(vif_plot))
ax.barh(y_pos, vif_plot['VIF'], color=bar_colors, alpha=0.85, edgecolor='black', linewidth=1.2)
ax.axvline(10, color=PALETTE['red'],    linewidth=2,   linestyle='--', alpha=0.8, label='VIF = 10')
ax.axvline(5,  color=PALETTE['orange'], linewidth=1.5, linestyle=':',  alpha=0.7, label='VIF = 5')
ax.set_yticks(y_pos); ax.set_yticklabels(vif_plot['Short'], fontsize=11, fontweight='bold')
ax.set_xlabel('Variance Inflation Factor (VIF)', fontsize=12, fontweight='bold')
ax.set_title('Multicollinearity Diagnostics — VIF by Feature', fontsize=14, fontweight='bold', pad=14)
plt.figtext(0.5, 0.86, 'VIF > 10 flagged in red  |  computed on training set (1995–2014)',
            ha='center', fontsize=10, color='#444')
for i, val in enumerate(vif_plot['VIF']):
    ax.text(min(val - 0.15, 10.8), i, f'{val:.1f}',
            va='center', ha='right', fontsize=9, fontweight='bold', color='black')
ax.set_xlim(0, 11)
ax.legend(fontsize=10, loc='lower right', framealpha=0.95)
ax.grid(axis='x', alpha=0.25, linestyle='--')
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.close("all")


# ── [6/8] 3-Panel Coefficient Comparison ──────────────────────────────────────
print("\n[6/8] 3-panel coefficient comparison...")
fig, axes = plt.subplots(1, 3, figsize=(22, 12))
for idx, mname in enumerate(['LASSO', 'Ridge', 'Elastic Net']):
    ax = axes[idx]
    coef_vals = models_level[mname].coef_.copy()
    abs_vals  = np.abs(coef_vals)
    excl_idx  = all_features.index(EXCLUDE) if EXCLUDE in all_features else None
    if excl_idx is not None: abs_vals[excl_idx] = -np.inf
    top15_idx  = np.argsort(abs_vals)[::-1][:15]
    top15_vals = coef_vals[top15_idx]
    colors     = [PALETTE['green'] if v > 0 else PALETTE['red'] for v in top15_vals]
    y_pos      = np.arange(len(top15_idx))
    ax.barh(y_pos, top15_vals, color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
    ax.axvline(0, color='black', linewidth=2.5, alpha=0.8, zorder=0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([shorten(all_features[i]) for i in top15_idx], fontsize=11, fontweight='bold')
    ax.set_xlabel('Coefficient', fontsize=12, fontweight='bold')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.25, linestyle='--')
    r2_tr = perf_level[perf_level['Model'] == mname]['Train R²'].values[0]
    r2_te = perf_level[perf_level['Model'] == mname]['Test R²'].values[0]
    n_sel = int(np.sum(models_level[mname].coef_ != 0))
    ax.set_title(f'{mname}\nTrain R²={r2_tr:.3f}  Test R²={r2_te:.3f}  |  {n_sel} features',
                 fontsize=13, fontweight='bold', pad=12)
    x_max = max(abs(top15_vals)) if len(top15_vals) else 1
    for i, val in enumerate(top15_vals):
        if abs(val) > 0.005:
            ax.text(abs(val) + x_max*0.03 if val > 0 else x_max*0.03, i, f'{val:+.3f}',
                    va='center', ha='left', fontsize=9.5, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.95, edgecolor='gray', linewidth=0.8))

fig.legend(handles=[Patch(facecolor=PALETTE['green'], label='Positive', alpha=0.85, edgecolor='black', linewidth=1.5),
                    Patch(facecolor=PALETTE['red'],   label='Negative', alpha=0.85, edgecolor='black', linewidth=1.5)],
           loc='lower center', bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=13, framealpha=0.98, edgecolor='black')
plt.suptitle('Standardised Coefficients — LASSO, Ridge, Elastic Net', fontsize=18, fontweight='bold', y=0.98)
plt.figtext(0.5, 0.94, 'Dep var: ECI  |  L1_ECI excluded  |  top 15 by absolute coefficient',
            ha='center', fontsize=12, color='#444')
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.close("all")


# ── [7/8] Model Agreement ─────────────────────────────────────────────────────
print("\n[7/8] Model agreement dot-and-range chart...")
imp_df = all_importance[all_importance['Feature'] != EXCLUDE].copy()
top12  = imp_df.sort_values('Elastic Net', ascending=False).head(12).reset_index(drop=True)
top12['Short'] = top12['Feature'].apply(shorten)
fig, ax = plt.subplots(figsize=(14, 8))
y_pos = np.arange(len(top12))
for i, row in top12.iterrows():
    vals = [row['LASSO'], row['Ridge'], row['Elastic Net']]
    ax.plot([min(vals), max(vals)], [i, i], color='#555555', alpha=0.45, linewidth=3, zorder=1)
    ax.scatter(row['LASSO'],       i, marker='o', s=160, alpha=0.92, color=PALETTE['red'],   edgecolor='black', linewidth=1.5, zorder=3, label='LASSO'       if i == 0 else '')
    ax.scatter(row['Ridge'],       i, marker='s', s=160, alpha=0.92, color=PALETTE['blue'],  edgecolor='black', linewidth=1.5, zorder=3, label='Ridge'       if i == 0 else '')
    ax.scatter(row['Elastic Net'], i, marker='^', s=160, alpha=0.92, color=PALETTE['green'], edgecolor='black', linewidth=1.5, zorder=3, label='Elastic Net' if i == 0 else '')
ax.set_yticks(y_pos); ax.set_yticklabels(top12['Short'], fontsize=11, fontweight='bold')
ax.set_xlabel('Normalised Feature Importance', fontsize=13, fontweight='bold')
ax.set_title('Model Agreement — LASSO, Ridge, Elastic Net', fontsize=15, fontweight='bold', pad=20)
ax.invert_yaxis()
ax.legend(fontsize=11, loc='lower right', framealpha=0.95, edgecolor='black')
ax.grid(axis='x', alpha=0.3, linestyle='--')
x_max_val = max(top12[['LASSO', 'Ridge', 'Elastic Net']].max())
ax.set_xlim(-0.02, x_max_val + 0.12)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.close("all")


# ── [8/8] Random Forest ───────────────────────────────────────────────────────
print("\n[8/8] Random Forest importance...")
rf_imp = pd.DataFrame({'Feature': all_features, 'Importance': rf.feature_importances_})
rf_top = (rf_imp[rf_imp['Feature'] != EXCLUDE]
          .sort_values('Importance', ascending=False).head(15).reset_index(drop=True))
norm_vals  = rf_top['Importance'] / rf_top['Importance'].max()
cmap       = plt.cm.YlOrRd
bar_colors = [cmap(0.35 + 0.60 * v) for v in norm_vals]
fig, ax = plt.subplots(figsize=(13, 9))
y_pos = np.arange(len(rf_top))
ax.barh(y_pos, rf_top['Importance'], color=bar_colors, edgecolor='black', linewidth=1.2, alpha=0.92)
ax.set_yticks(y_pos); ax.set_yticklabels([shorten(f) for f in rf_top['Feature']], fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=12, fontweight='bold')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.25, linestyle='--')
ax.set_xlim(right=rf_top['Importance'].max() * 1.22)
x_max_rf = rf_top['Importance'].max()
for i, val in enumerate(rf_top['Importance']):
    ax.text(val + x_max_rf*0.012, i, f'{val:.4f}', va='center', ha='left', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.88, edgecolor='gray', linewidth=0.6))
ax.set_title(f'Random Forest — Feature Importance\nOOB R²={rf.oob_score_:.3f}  |  200 trees  |  max_depth=4',
             fontsize=13, fontweight='bold', pad=14)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=x_max_rf))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.025, pad=0.02)
cbar.set_label('Importance', fontsize=10)
plt.tight_layout(rect=[0, 0, 0.97, 0.92])
plt.close("all")
print(f"\n✓ All 8 static charts saved to {OUT}/")

# ## 6. Visualisations — Interactive (Plotly)
# 
# Five interactive charts matching the static charts above.

# ── [A] Train vs Test R² ─────────────────────────────────────────────────────
print("[A] OOS R² comparison...")
figA = make_subplots(rows=1, cols=2, subplot_titles=['Target: ECI', 'Target: ΔECI'],
                     horizontal_spacing=0.1)
for col, perf in enumerate([perf_level, perf_delta], 1):
    figA.add_trace(go.Bar(name='Train R²', x=perf['Model'], y=perf['Train R²'],
                          marker_color=PALETTE['blue'], opacity=0.85,
                          text=[f'{v:.3f}' for v in perf['Train R²']], textposition='outside',
                          showlegend=(col==1)), row=1, col=col)
    figA.add_trace(go.Bar(name='Test R²',  x=perf['Model'], y=perf['Test R²'],
                          marker_color=PALETTE['red'],  opacity=0.85,
                          text=[f'{v:.3f}' for v in perf['Test R²']],  textposition='outside',
                          showlegend=(col==1)), row=1, col=col)
    figA.update_yaxes(title_text='R²' if col==1 else '',
                      gridcolor=STYLE['grid_color'], row=1, col=col)
figA.update_layout(**base_layout(barmode='group', height=STYLE['chart_height'],
                                  legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center',
                                              font=dict(size=STYLE['legend_size'])),
                                  margin=dict(l=60, r=40, t=80, b=80)))
save_chart(figA, os.path.join(OUT, 'OOS_R2_interactive'))


# ── [B] Prediction intervals ──────────────────────────────────────────────────
print("\n[B] Prediction intervals...")
plot_s  = interval_df.sort_values('Actual').reset_index(drop=True)
xlabels = plot_s.apply(lambda r: f"{r['Country Code']} {int(r['Year'])}", axis=1)

figB = go.Figure()
figB.add_trace(go.Scatter(
    x=list(range(len(plot_s))) + list(range(len(plot_s)))[::-1],
    y=plot_s['Q90'].tolist() + plot_s['Q10'].tolist()[::-1],
    fill='toself', fillcolor='rgba(74,111,165,0.2)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', name='80% band'))
figB.add_trace(go.Scatter(x=list(range(len(plot_s))), y=plot_s['Q50'],
    mode='lines', line=dict(color=PALETTE['blue'], width=2), name='Median (Q50)'))
for mask, color, sym, lbl in [
    (plot_s['In_Band'],  PALETTE['green'], 'circle',  'In band'),
    (~plot_s['In_Band'], PALETTE['red'],   'diamond', 'Out of band'),
]:
    figB.add_trace(go.Scatter(
        x=plot_s.index[mask].tolist(), y=plot_s.loc[mask, 'Actual'],
        mode='markers', marker=dict(color=color, size=7 if lbl=='In band' else 9, symbol=sym, opacity=0.85),
        name=f'Actual ({lbl})',
        hovertemplate='%{text}<br>Actual: %{y:.3f}<extra></extra>',
        text=xlabels[mask].tolist()))
figB.update_layout(**base_layout(
    xaxis=dict(title='Observations (sorted by actual)', showticklabels=False,
               gridcolor=STYLE['grid_color']),
    yaxis=dict(title='ECI', gridcolor=STYLE['grid_color']),
    annotations=[dict(text=f"Coverage: {coverage:.1%}  |  Avg width: {avg_width:.4f}",
                      xref='paper', yref='paper', x=0.01, y=0.99, showarrow=False,
                      font=dict(size=11, color='#555'),
                      bgcolor='rgba(250,250,250,0.9)', bordercolor='#ddd', borderwidth=1, borderpad=6)],
    legend=dict(font=dict(size=STYLE['legend_size']), bgcolor='rgba(250,250,250,0.9)',
                bordercolor='#e5e7eb', borderwidth=1)))
save_chart(figB, os.path.join(OUT, 'PredictionIntervals_interactive'))


# ── [C] VIF interactive ───────────────────────────────────────────────────────
print("\n[C] VIF chart (interactive)...")
vif_plot = vif_data.copy()
vif_plot['Short'] = vif_plot['Feature'].apply(shorten)
vif_plot = vif_plot.sort_values('VIF', ascending=True).reset_index(drop=True)
colors_vif = [PALETTE['red'] if v > 10 else PALETTE['blue'] for v in vif_plot['VIF']]

figC = go.Figure(go.Bar(
    y=vif_plot['Short'], x=vif_plot['VIF'], orientation='h',
    marker=dict(color=colors_vif, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.1f}' for v in vif_plot['VIF']], textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
))
figC.add_vline(x=10, line=dict(color=PALETTE['red'],    width=2,   dash='dash'),
               annotation_text='VIF = 10', annotation_position='right',
               annotation_font=dict(size=STYLE['annotation_size'], color=PALETTE['red']))
figC.add_vline(x=5,  line=dict(color=PALETTE['orange'], width=1.5, dash='dot'),
               annotation_text='VIF = 5',  annotation_position='right',
               annotation_font=dict(size=STYLE['annotation_size'], color=PALETTE['orange']))
figC.update_layout(**base_layout(height=STYLE['chart_height_tall'], margin=STYLE['margin_bar'],
                                  xaxis=dict(title=dict(text='Variance Inflation Factor',
                                                        font=dict(size=STYLE['axis_title_size'])),
                                             gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
                                             range=[0, max(vif_plot['VIF'].max()*1.15, 12)]),
                                  yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
                                  showlegend=False))
save_chart(figC, os.path.join(OUT, 'VIF_model2_resource_rich_interactive'), width=1100, height=700)


# ── [D] Model Agreement interactive ──────────────────────────────────────────
print("\n[D] Model agreement (interactive)...")
top12_r = top12.iloc[::-1].reset_index(drop=True)
figD = go.Figure()
for _, row in top12_r.iterrows():
    vals = [row['LASSO'], row['Ridge'], row['Elastic Net']]
    figD.add_trace(go.Scatter(x=[min(vals), max(vals)], y=[row['Short'], row['Short']],
                               mode='lines', line=dict(color='#aab0b8', width=3),
                               showlegend=False, hoverinfo='skip'))
for mname, sym, col in [('LASSO','circle',PALETTE['red']),('Ridge','square',PALETTE['blue']),('Elastic Net','triangle-up',PALETTE['green'])]:
    figD.add_trace(go.Scatter(x=top12_r[mname], y=top12_r['Short'], mode='markers',
                               marker=dict(symbol=sym, size=12, color=col, line=dict(color='#1a2744', width=1)),
                               name=mname, hovertemplate='%{y}: %{x:.3f}<extra>' + mname + '</extra>'))
figD.update_layout(**base_layout(height=STYLE['chart_height'], margin=STYLE['margin_bar'],
                                  xaxis=dict(title=dict(text='Normalised Feature Importance',
                                                        font=dict(size=STYLE['axis_title_size'])),
                                             gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
                                             range=[-0.02, top12[['LASSO','Ridge','Elastic Net']].max().max()+0.08]),
                                  yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
                                  legend=dict(font=dict(size=STYLE['legend_size']),
                                              yanchor='bottom', y=0.02, xanchor='right', x=0.98,
                                              bgcolor='rgba(250,250,250,0.95)', bordercolor='#e5e7eb', borderwidth=1)))
save_chart(figD, os.path.join(OUT, 'ModelAgreement_model2_resource_rich_interactive'), width=1100, height=600)


# ── [E] RF importance interactive ─────────────────────────────────────────────
print("\n[E] RF importance (interactive)...")
def lerp_color(t, c_lo=(230,185,128), c_hi=(180,80,40)):
    return f'rgb({int(c_lo[0]+(c_hi[0]-c_lo[0])*t)},{int(c_lo[1]+(c_hi[1]-c_lo[1])*t)},{int(c_lo[2]+(c_hi[2]-c_lo[2])*t)})'

rf_top_r       = rf_top.iloc[::-1].reset_index(drop=True)
norm_vals_r    = (rf_top_r['Importance'] / rf_top_r['Importance'].max()).values
bar_colors_rf  = [lerp_color(v) for v in norm_vals_r]

figE = go.Figure(go.Bar(
    y=[shorten(f) for f in rf_top_r['Feature']], x=rf_top_r['Importance'], orientation='h',
    marker=dict(color=bar_colors_rf, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.4f}' for v in rf_top_r['Importance']], textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
))
figE.update_layout(**base_layout(height=STYLE['chart_height_tall'], margin=STYLE['margin_bar'],
                                  xaxis=dict(title=dict(text='Mean Decrease in Impurity',
                                                        font=dict(size=STYLE['axis_title_size'])),
                                             gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
                                             range=[0, rf_top['Importance'].max()*1.22]),
                                  yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
                                  showlegend=False,
                                  annotations=[dict(text=f"OOB R²={rf.oob_score_:.3f}  |  200 trees  |  max_depth=4",
                                                    xref='paper', yref='paper', x=0.98, y=0.02,
                                                    showarrow=False, font=dict(size=STYLE['annotation_size'], color='#666'),
                                                    bgcolor='rgba(250,250,250,0.95)', bordercolor='#e5e7eb',
                                                    borderwidth=1, borderpad=6)]))
save_chart(figE, os.path.join(OUT, 'RF_model2_resource_rich_interactive'), width=1100, height=700)
print(f"\n✓ All interactive charts saved to {OUT}/")

# ## 7. Coefficient Summary Table
# 
# Heatmap-style table: signed coefficients, normalised importance, VIF, and selection flags
# for LASSO and Elastic Net.

# ── Build table ───────────────────────────────────────────────────────────────
table = pd.DataFrame({'Feature': all_features})
for mname, model in lin_models.items():
    table[mname] = model.coef_
for mname in ['LASSO', 'Ridge', 'Elastic Net']:
    table[f'{mname}_Imp'] = all_importance.set_index('Feature')[mname].reindex(all_features).values
table = table.merge(vif_data, on='Feature')
table['LASSO_Selected']       = (table['LASSO']       != 0).astype(int)
table['Elastic Net_Selected'] = (table['Elastic Net'] != 0).astype(int)
table = (table[table['Feature'] != EXCLUDE]
         .assign(sort_key=lambda d: d['Elastic Net'].abs())
         .sort_values('sort_key', ascending=False)
         .drop(columns='sort_key')
         .reset_index(drop=True))
table['Feature'] = table['Feature'].apply(shorten)
table.to_csv(os.path.join(OUT, 'coefficient_summary_table.csv'), index=False)
print(f"  ✓ {OUT}/coefficient_summary_table.csv")

# ── Console summary ────────────────────────────────────────────────────────────
print(f"\n{'Feature':<24} {'LASSO':>8} {'Ridge':>8} {'E-Net':>8} "
      f"{'L-Imp':>7} {'R-Imp':>7} {'EN-Imp':>7} {'VIF':>6} {'L-Sel':>6} {'EN-Sel':>6}")
print("─" * 95)
for _, row in table.iterrows():
    print(f"{row['Feature']:<24} {row['LASSO']:>+8.3f} {row['Ridge']:>+8.3f} {row['Elastic Net']:>+8.3f} "
          f"{row['LASSO_Imp']:>7.3f} {row['Ridge_Imp']:>7.3f} {row['Elastic Net_Imp']:>7.3f} "
          f"{row['VIF']:>6.2f} {'✓' if row['LASSO_Selected'] else '':>6} {'✓' if row['Elastic Net_Selected'] else '':>6}")

# ── Heatmap chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
ax.axis('off')
col_labels = ['Feature', 'LASSO\nCoef', 'Ridge\nCoef', 'E-Net\nCoef',
              'LASSO\nImp', 'Ridge\nImp', 'E-Net\nImp', 'VIF', 'LASSO\nSel', 'EN\nSel']
cell_vals = [[
    row['Feature'], f"{row['LASSO']:+.3f}", f"{row['Ridge']:+.3f}", f"{row['Elastic Net']:+.3f}",
    f"{row['LASSO_Imp']:.3f}", f"{row['Ridge_Imp']:.3f}", f"{row['Elastic Net_Imp']:.3f}",
    f"{row['VIF']:.2f}", '✓' if row['LASSO_Selected'] else '', '✓' if row['Elastic Net_Selected'] else '',
] for _, row in table.iterrows()]

tbl = ax.table(cellText=cell_vals, colLabels=col_labels, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.0, 1.6)
for col_idx in range(len(col_labels)):
    tbl[0, col_idx].set_facecolor('#2C3E50')
    tbl[0, col_idx].set_text_props(color='white', fontweight='bold', fontsize=9)
for row_idx, (_, row) in enumerate(table.iterrows(), start=1):
    tbl[row_idx, 0].set_facecolor('#ECF0F1' if row_idx % 2 else '#D5D8DC')
    tbl[row_idx, 0].set_text_props(fontweight='bold', ha='left')
    for ci, cn in [(1,'LASSO'),(2,'Ridge'),(3,'Elastic Net')]:
        val = row[cn]
        tbl[row_idx, ci].set_facecolor('#D5F5E3' if val > 0.005 else '#FADBD8' if val < -0.005 else '#FDFEFE')
    for ci, cn in [(4,'LASSO_Imp'),(5,'Ridge_Imp'),(6,'Elastic Net_Imp')]:
        alpha = 0.15 + 0.70 * row[cn]
        tbl[row_idx, ci].set_facecolor(mcolors.to_rgba(PALETTE['blue'], alpha=alpha))
    vif_val = row['VIF']
    if vif_val > 10:
        tbl[row_idx, 7].set_facecolor('#FADBD8'); tbl[row_idx, 7].set_text_props(color='#C0392B', fontweight='bold')
    elif vif_val > 5:
        tbl[row_idx, 7].set_facecolor('#FDEBD0'); tbl[row_idx, 7].set_text_props(color='#E67E22', fontweight='bold')
    for ci, sc in [(8,'LASSO_Selected'),(9,'Elastic Net_Selected')]:
        if row[sc]:
            tbl[row_idx, ci].set_facecolor('#D5F5E3'); tbl[row_idx, ci].set_text_props(color='#1E8449', fontweight='bold')

ax.set_title('Summary Table — LASSO, Ridge, Elastic Net\nCoefficients, Importance, VIF, Selection',
             fontsize=14, fontweight='bold', pad=16, y=0.98)
plt.figtext(0.5, 0.01, 'Green=positive  |  Red=negative  |  Blue intensity=importance  |  Orange VIF=multicollinearity  |  ✓=selected',
            ha='center', fontsize=9, color='#444')
plt.tight_layout()
plt.close("all")

# ## 8. Forecast 2020–2030 & Country Rankings
# 
# **Strategy:**
# 1. Retrain all models on the full 1995–2019 sample.
# 2. Extrapolate features using the last-5-year linear trend per country.
# 3. Predict iteratively (each year's L1_ECI = prior year's prediction).
# 4. Ensemble = mean of all level and accumulated-Δ predictions.
# 
# Results are saved to `Final/NB5/ECI_Forecast_2020_2030.csv` and
# `Final/NB5/Country_Ranking_2020_2030.csv`.

# ── Retrain on full 1995–2019 sample ──────────────────────────────────────────
X_full_raw    = df[all_features].values
y_full_level  = df['Economic Complexity Index'].values
y_full_delta  = df['ECI_delta'].values

scaler_full = StandardScaler()
X_full      = scaler_full.fit_transform(X_full_raw)

full_years = df['Year'].values
tscv_full  = PanelTemporalCV(full_years, n_splits=5, gap=1, min_train_years=8)

models_full_level = fit_all_models(X_full, y_full_level, tscv_full,
                                   label='ECI (full)', years=full_years)
models_full_delta = fit_all_models(X_full, y_full_delta, tscv_full,
                                   label='ΔECI (full)', years=full_years)
print("\n✓ All models retrained on full 1995–2019 sample.")

# ── Build future feature rows by linear extrapolation ────────────────────────
FORECAST_YEARS = list(range(2020, 2031))
trend_features = [f for f in base_features if f not in ['L1_ECI', 'Landlocked']]

def extrapolate_country(country_df, years):
    last5 = country_df.tail(5)
    rows = []
    for yr in years:
        row = {'Year': yr, 'Country Code': country_df['Country Code'].iloc[0],
               'Country Name': country_df['Country Name'].iloc[0]}
        for feat in trend_features:
            vals = last5[feat].dropna().values
            if len(vals) >= 2:
                x_t = np.arange(len(vals))
                slope, intercept = np.polyfit(x_t, vals, 1)
                steps = yr - int(last5['Year'].iloc[-1])
                row[feat] = float(intercept + slope * (len(vals) - 1 + steps))
            else:
                row[feat] = float(last5[feat].iloc[-1]) if len(vals) == 1 else 0.0
        row['Landlocked'] = float(country_df['Landlocked'].iloc[0])
        rows.append(row)
    return rows

future_rows = []
for cc, cdf in df.groupby('Country Code'):
    future_rows.extend(extrapolate_country(cdf, FORECAST_YEARS))
future_df = pd.DataFrame(future_rows)

# ── Forecast uncertainty note ────────────────────────────────────────────────
# This forecast uses linear extrapolation of features over 11 years (2020-2030)
# from the last 5 observed data points. Key limitations:
#   1. Features like IMF credit, inflation, or political stability can reverse
#      sharply — a linear trend assumption is strong over this horizon.
#   2. The iterative loop compounds errors: each year's L1_ECI = prior year's
#      prediction, so small biases accumulate over 11 steps.
#   3. No prediction intervals are attached to the ensemble forecast below.
#      The quantile GB models (gb_q10/q50/q90) were fit on the train-only sample
#      and their intervals are reported in the OOS section, not here.
# These limitations should be stated prominently in the report narrative.
# ─────────────────────────────────────────────────────────────────────────────

# ── Iterative forecast ────────────────────────────────────────────────────────
records = []
for cc, cdf in df.groupby('Country Code'):
    last_eci    = float(cdf.sort_values('Year')['Economic Complexity Index'].iloc[-1])
    future_sub  = future_df[future_df['Country Code'] == cc].sort_values('Year')

    running_eci_level = {n: last_eci for n in models_full_level}
    running_eci_delta = {n: last_eci for n in models_full_delta}

    for _, frow in future_sub.iterrows():
        preds_level, preds_delta = {}, {}
        for name, model in models_full_level.items():
            row_data = frow.copy()
            row_data['L1_ECI'] = running_eci_level[name]
            row_data['HCI_x_ProductionValue']       = (row_data['Human capital index'] - _hci_mean) * \
                (row_data['Total_Production_Value_Per_Capita'] - _prod_mean)
            row_data['RuleOfLaw_x_ProductionValue'] = (row_data['Rule of law index']   - _rol_mean) * \
                (row_data['Total_Production_Value_Per_Capita'] - _prod_mean)
            x_vec   = np.array([row_data[f] for f in all_features]).reshape(1, -1)
            pred    = model.predict(scaler_full.transform(x_vec))[0]
            preds_level[name] = pred
            running_eci_level[name] = pred
        for name, model in models_full_delta.items():
            row_data = frow.copy()
            row_data['L1_ECI'] = running_eci_delta[name]
            row_data['HCI_x_ProductionValue']       = (row_data['Human capital index'] - _hci_mean) * \
                (row_data['Total_Production_Value_Per_Capita'] - _prod_mean)
            row_data['RuleOfLaw_x_ProductionValue'] = (row_data['Rule of law index']   - _rol_mean) * \
                (row_data['Total_Production_Value_Per_Capita'] - _prod_mean)
            x_vec   = np.array([row_data[f] for f in all_features]).reshape(1, -1)
            delta_p = model.predict(scaler_full.transform(x_vec))[0]
            accumulated = running_eci_delta[name] + delta_p
            preds_delta[f'{name} (Δ)'] = accumulated
            running_eci_delta[name] = accumulated

        all_preds = {**preds_level, **preds_delta}
        rec = {'Country Code': cc, 'Country Name': frow['Country Name'],
               'Year': frow['Year'], 'Last_Known_ECI': last_eci,
               'Ensemble': np.mean(list(all_preds.values()))}
        rec.update(all_preds)
        records.append(rec)

forecast_df = pd.DataFrame(records)

country_summary = forecast_df.groupby('Country Code').agg(
    Country=('Country Name', 'first'),
    ECI_2019=('Last_Known_ECI', 'first'),
    ECI_2030_Ensemble=('Ensemble', 'last'),
).reset_index()
country_summary['Total_Change']      = country_summary['ECI_2030_Ensemble'] - country_summary['ECI_2019']
country_summary['Annualised_Change'] = country_summary['Total_Change'] / 11
country_summary['Pct_Change']        = (country_summary['Total_Change'] /
                                         country_summary['ECI_2019'].abs().clip(lower=0.01) * 100)
country_summary = country_summary.sort_values('Total_Change', ascending=False).reset_index(drop=True)

print("\n══ Top 10 Countries by Predicted ECI Improvement (2020–2030) ═══════")
print(country_summary[['Country','ECI_2019','ECI_2030_Ensemble','Total_Change','Annualised_Change']].head(10).to_string(index=False))
print("\n══ Bottom 10 ══════════════════════════════════════════════════════════")
print(country_summary[['Country','ECI_2019','ECI_2030_Ensemble','Total_Change','Annualised_Change']].tail(10).to_string(index=False))

# ── Forecast charts ───────────────────────────────────────────────────────────
print("[Forecast 1/4] Country ranking chart...")
top10    = country_summary.head(10)
bottom10 = country_summary.tail(10).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, sub, title in [(axes[0], top10, 'Top 10 Improvers (2020–2030)'),
                        (axes[1], bottom10, 'Bottom 10 Decliners (2020–2030)')]:
    col = [PALETTE['green'] if v > 0 else PALETTE['red'] for v in sub['Total_Change']]
    y_pos = np.arange(len(sub))
    ax.barh(y_pos, sub['Total_Change'], color=col, edgecolor='black', linewidth=1, alpha=0.88)
    ax.set_yticks(y_pos); ax.set_yticklabels(sub['Country'], fontsize=10, fontweight='bold')
    ax.axvline(0, color='black', linewidth=1.2, alpha=0.6)
    ax.set_xlabel('ΔECI 2019→2030 (ensemble)', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    for i, val in enumerate(sub['Total_Change']):
        ax.text(val + (0.005 if val > 0 else -0.005), i, f'{val:+.3f}',
                va='center', ha='left' if val > 0 else 'right', fontsize=9, fontweight='bold')

plt.suptitle('ECI Forecast 2020–2030 — Country Rankings', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.close("all")


print("\n[Forecast 2/4] Top improvers trajectory...")
top5_codes = country_summary.head(5)['Country Code'].tolist()
hist_top5  = df[df['Country Code'].isin(top5_codes)][['Country Code','Country Name','Year','Economic Complexity Index']].copy()
fc_top5    = forecast_df[forecast_df['Country Code'].isin(top5_codes)][['Country Code','Year','Ensemble']].rename(columns={'Ensemble':'Economic Complexity Index'})
fc_top5['Country Name'] = fc_top5['Country Code'].map(dict(zip(hist_top5['Country Code'], hist_top5['Country Name'])))

fig, ax = plt.subplots(figsize=(14, 7))
cmap_fc = plt.get_cmap('tab10')
for i, cc in enumerate(top5_codes):
    col   = cmap_fc(i)
    cname = country_summary[country_summary['Country Code']==cc]['Country'].values[0]
    h = hist_top5[hist_top5['Country Code']==cc].sort_values('Year')
    f = fc_top5[fc_top5['Country Code']==cc].sort_values('Year')
    ax.plot(h['Year'], h['Economic Complexity Index'], color=col, linewidth=2, label=cname)
    ax.plot(f['Year'], f['Economic Complexity Index'], color=col, linewidth=2, linestyle='--', alpha=0.7)
ax.axvline(2019.5, color='grey', linewidth=1.5, linestyle='--', alpha=0.8, label='Forecast start')
ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Economic Complexity Index', fontsize=12, fontweight='bold')
ax.set_title('ECI Trajectories — Top 5 Forecast Improvers', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper left', framealpha=0.95); ax.grid(alpha=0.2, linestyle='--')
plt.tight_layout()
plt.close("all")


print("\n[Forecast 3/4] Heatmap (all countries)...")
cnames_map = df[['Country Code','Country Name']].drop_duplicates()
code_to_name = dict(zip(cnames_map['Country Code'], cnames_map['Country Name']))

hist_pivot = df.pivot_table(index='Country Code', columns='Year',
                             values='Economic Complexity Index', aggfunc='mean')
fc_pivot   = forecast_df.pivot_table(index='Country Code', columns='Year',
                                      values='Ensemble', aggfunc='mean')
full_pivot  = pd.concat([hist_pivot, fc_pivot], axis=1)
full_pivot  = full_pivot.loc[:, ~full_pivot.columns.duplicated(keep='last')]
full_pivot.columns = [int(c) for c in full_pivot.columns]
full_pivot  = full_pivot[sorted(full_pivot.columns)]
full_pivot  = full_pivot.loc[country_summary.sort_values('ECI_2030_Ensemble', ascending=True)['Country Code']]
full_pivot.index = [code_to_name.get(c, c) for c in full_pivot.index]

fig_heat = go.Figure(data=go.Heatmap(
    z=full_pivot.values, x=full_pivot.columns.tolist(), y=full_pivot.index.tolist(),
    colorscale='RdYlGn',
    colorbar=dict(title=dict(text='ECI', font=dict(size=11))),
    hovertemplate='%{y}<br>Year: %{x}<br>ECI: %{z:.3f}<extra></extra>',
))
fig_heat.add_vline(x=2019.5, line=dict(color='white', width=3))
fig_heat.update_layout(**base_layout(
    height=max(700, len(full_pivot)*16),
    margin=dict(l=140, r=40, t=20, b=60),
    xaxis=dict(title='Year', dtick=2, tickfont=dict(size=9), type='linear'),
    yaxis=dict(tickfont=dict(size=8)),
))
os.makedirs(OUT_CHARTS, exist_ok=True)
fig_heat.write_html(os.path.join(OUT_CHARTS, '13_ml__eci_forecast_heatmap_all_54_countries.html'), config=WRITE_CONFIG)
print('  Saved: 13_ml__eci_forecast_heatmap_all_54_countries.html')



# ── CHART: Average ECI Change 2015–2019 — Actual vs Predicted by Country ──────
print("\n[Forecast 4/4] Heatmap (all countries)...")

# Predictions from best ΔECI model
best_delta_name = perf_delta.iloc[0]['Model']
test_df_plot = test_df.copy().reset_index(drop=True)
test_df_plot['Pred_Delta'] = models_delta[best_delta_name].predict(X_test)

# Average actual and predicted ΔECI per country
country_avg = test_df_plot.groupby('Country Code').agg(
    Country=('Country Name', 'first'),
    Actual_Avg_Delta=('ECI_delta', 'mean'),
    Pred_Avg_Delta=('Pred_Delta', 'mean'),
).reset_index()

# ── Biggest mismatches: actual vs predicted ───────────────────────────────────
country_avg['Residual'] = country_avg['Actual_Avg_Delta'] - country_avg['Pred_Avg_Delta']
country_avg['Abs_Residual'] = country_avg['Residual'].abs()
mismatches = country_avg.sort_values('Abs_Residual', ascending=False).reset_index(drop=True)

print("\n══ Biggest Mismatches — Actual vs Predicted Avg ΔECI (2015–2019) ═══════")
print(mismatches[['Country Code', 'Country',
                   'Actual_Avg_Delta', 'Pred_Avg_Delta',
                   'Residual']].head(15).to_string(index=False))

r2_val  = r2_score(country_avg['Actual_Avg_Delta'], country_avg['Pred_Avg_Delta'])
rmse_val = np.sqrt(mean_squared_error(country_avg['Actual_Avg_Delta'], country_avg['Pred_Avg_Delta']))

fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(country_avg['Actual_Avg_Delta'], country_avg['Pred_Avg_Delta'],
           s=50, color=PALETTE['blue'], alpha=0.75, edgecolor='black', linewidth=0.5)

# Label each point with country code
for _, row in country_avg.iterrows():
    ax.annotate(row['Country Code'], (row['Actual_Avg_Delta'], row['Pred_Avg_Delta']),
                fontsize=7, ha='left', va='bottom', color='#333',
                xytext=(4, 4), textcoords='offset points')

# 45-degree line
lims = [min(country_avg['Actual_Avg_Delta'].min(), country_avg['Pred_Avg_Delta'].min()) - 0.02,
        max(country_avg['Actual_Avg_Delta'].max(), country_avg['Pred_Avg_Delta'].max()) + 0.02]
ax.plot(lims, lims, '--', color=PALETTE['red'], linewidth=1.5, alpha=0.8, label='45° line')
ax.set_xlim(lims)
ax.set_ylim(lims)

# Trend line (OLS fit)
z = np.polyfit(country_avg['Actual_Avg_Delta'], country_avg['Pred_Avg_Delta'], 1)
p = np.poly1d(z)
x_trend = np.linspace(lims[0], lims[1], 100)
ax.plot(x_trend, p(x_trend), '-', color=PALETTE['green'], linewidth=2, alpha=0.8,
        label=f'OLS fit (slope={z[0]:.2f})')

# Zero reference lines
ax.axhline(0, color='black', linewidth=0.5, alpha=0.3)
ax.axvline(0, color='black', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Actual Average ΔECI (2015–2019)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Average ΔECI (2015–2019)', fontsize=12, fontweight='bold')
ax.set_title(f'{best_delta_name} — Country-Level Average ΔECI\n',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.close("all")
# ── Export CSV ─────────────────────────────────────────────────────────────────
print("\n[Forecast 4/4] Exporting CSVs...")
forecast_df.to_csv(os.path.join(OUT, 'ECI_Forecast_2020_2030.csv'), index=False)
country_summary.to_csv(os.path.join(OUT, 'Country_Ranking_2020_2030.csv'), index=False)
print(f"  ✓ {OUT}/ECI_Forecast_2020_2030.csv  ({len(forecast_df):,} rows)")
print(f"  ✓ {OUT}/Country_Ranking_2020_2030.csv ({len(country_summary)} rows)")
print(f"\n✓ All forecast charts saved to {OUT}/")

# ## 9. Summary
# 
# | Item | Value |
# |---|---|
# | Sample | 54 countries × 25 years |
# | Train / Test | 1995–2014 / 2015–2019 |
# | Features | 23 (18 base + 3 rolling/HHI + 2 interactions) |
# | Models | LASSO, Ridge, Elastic Net, RF (+ XGBoost if installed) |
# | Targets | ECI level + ΔECI (year-on-year) |
# | SE fix | Scaler fit on train only |

print("=" * 65)
print("  NB5 SUMMARY — Supervised ML: Drivers of ECI")
print("=" * 65)
print(f"  Sample       : {df['Country Code'].nunique()} countries × {df['Year'].nunique()} years = {len(df):,} obs")
print(f"  Train / Test : {len(train_df):,} obs (1995–{TRAIN_END}) / {len(test_df):,} obs ({TEST_START}–2019)")
print(f"  Features     : {len(all_features)} ({len(base_features)} base + {len(interaction_features)} interactions)")
print()
print("  OOS performance — ECI:")
for _, row in perf_level.iterrows():
    print(f"    {row['Model']:<14}  Train R²={row['Train R²']:.4f}  Test R²={row['Test R²']:.4f}  "
          f"Gap={row['Overfit Gap']:+.4f}  Test RMSE={row['Test RMSE']:.4f}")
print()
print("  OOS performance — ΔECI:")
for _, row in perf_delta.iterrows():
    print(f"    {row['Model']:<14}  Train R²={row['Train R²']:.4f}  Test R²={row['Test R²']:.4f}  "
          f"Gap={row['Overfit Gap']:+.4f}  Test RMSE={row['Test RMSE']:.4f}")
print()
print(f"  80% PI coverage  : {coverage:.1%}  (avg width: {avg_width:.4f})")
print(f"  LASSO selected   : {lasso_sel}/{len(all_features)} features")
print(f"  EN selected      : {en_sel}/{len(all_features)} features")
print(f"  Top 5 (EN imp.)  : {top5}")
if HAS_SHAP and shap_results:
    for mname, sv in shap_results.items():
        ma = np.abs(sv).mean(axis=0)
        fi = [i for i, f in enumerate(all_features) if f != EXCLUDE]
        t3 = np.argsort(ma[fi])[::-1][:3]
        print(f"  SHAP top 3 ({mname}): {[short_names[fi[i]] for i in t3]}")
print()
print(f"  Outputs saved to: {OUT}/")
print("=" * 65)

## Section 3 — ML Charts: 07–13

Reads pre-computed CSVs from `Final/NB5/` (written by Section 2) and produces the final HTML charts:

- **07**: Feature importance consensus (LASSO / Ridge / Elastic Net)
- **08**: Standardised coefficients
- **09**: Train vs Test R²
- **10**: Actual vs Predicted (ECI + ΔECI)
- **11**: ECI forecast — top/bottom improvers
- **13**: ECI forecast heatmap (all 54 countries)

In [ ]:
# Note: all imports and style constants are already loaded from the Setup cell.
# OUT, NB5 variables are set above.
# viz_utils functions (save, base_layout, PALETTE, etc.) are imported.

def _hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

LABEL_EXCL = ['L1_ECI', 'Inflation_roll5', 'RealRate_roll5', 'Resource_HHI']


# CHART 07 — Feature Importance Consensus (LASSO / Ridge / Elastic Net)
# =============================================================================
print("\n[07] Feature importance consensus...")

imp = pd.read_csv(os.path.join(NB5, 'all_importance.csv'))
imp = imp[~imp['Feature'].apply(lambda f: any(e in f for e in LABEL_EXCL))]

lin_cols = [c for c in ['LASSO', 'Ridge', 'Elastic Net'] if c in imp.columns]
imp = (imp.sort_values('Elastic Net' if 'Elastic Net' in imp.columns else lin_cols[0],
                       ascending=False)
         .head(12).reset_index(drop=True))
imp = imp.iloc[::-1].reset_index(drop=True)
imp['Label'] = imp['Feature'].apply(shorten_feat)

fig = go.Figure()
for _, row in imp.iterrows():
    vals = [row[c] for c in lin_cols if not pd.isna(row[c])]
    if len(vals) >= 2:
        fig.add_trace(go.Scatter(
            x=[min(vals), max(vals)], y=[row['Label'], row['Label']],
            mode='lines', line=dict(color='#c0c8d4', width=3),
            showlegend=False, hoverinfo='skip',
        ))

model_cfg07 = [
    ('LASSO',       'circle',      PALETTE['lasso']),
    ('Ridge',       'square',      PALETTE['ridge']),
    ('Elastic Net', 'triangle-up', PALETTE['en']),
]
for mname, sym, col in model_cfg07:
    if mname not in imp.columns:
        continue
    fig.add_trace(go.Scatter(
        x=imp[mname], y=imp['Label'],
        mode='markers',
        marker=dict(symbol=sym, size=13, color=col, line=dict(color='white', width=1.5)),
        name=mname,
        hovertemplate=f'%{{y}}: %{{x:.3f}}<extra>{mname}</extra>',
    ))

x_max = imp[lin_cols].max().max()
fig.update_layout(**base_layout(
    height=560,
    margin=dict(l=200, r=80, t=70, b=80),
    xaxis=dict(title=dict(text='Normalised Feature Importance (min-max, 0–1)', font=dict(size=13)),
               range=[-0.02, x_max + 0.1], gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig, '07_ml__feature_importance_consensus_three_models', OUT, w=1100, h=560)


# =============================================================================
# CHART 08 — Standardised Coefficients (LASSO / Ridge / Elastic Net)
# =============================================================================
print("\n[08] Standardised coefficients...")

tbl = pd.read_csv(os.path.join(NB5, 'coefficient_summary_table.csv'))
tbl = tbl[~tbl['Feature'].apply(lambda f: any(e in f for e in LABEL_EXCL))]
tbl['abs_en'] = tbl['Elastic Net'].abs()
top = tbl.nlargest(12, 'abs_en').sort_values('abs_en', ascending=True).reset_index(drop=True)

fig = go.Figure()
fig.add_vline(x=0, line=dict(color='#444', width=1.5))

model_cfg08 = [
    ('LASSO',       PALETTE['lasso']),
    ('Ridge',       PALETTE['ridge']),
    ('Elastic Net', PALETTE['en']),
]
for mname, col in model_cfg08:
    if mname not in top.columns:
        continue
    fig.add_trace(go.Bar(
        y=top['Feature'], x=top[mname], orientation='h',
        name=mname,
        marker=dict(color=col, opacity=0.88, line=dict(color='white', width=0.5)),
        hovertemplate=f'%{{y}}: %{{x:+.3f}}<extra>{mname}</extra>',
    ))

fig.update_layout(**base_layout(
    barmode='group', height=620,
    margin=dict(l=200, r=80, t=70, b=60),
    xaxis=dict(title=dict(text='Coefficient (standardised inputs)', font=dict(size=13)),
               gridcolor=GRID, gridwidth=0.5, zeroline=False),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig, '08_ml__standardised_coefficients_lasso_ridge_en', OUT, w=1100, h=620)

def _hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


# =============================================================================
# CHART 09 — Train vs Test R² (XGBoost removed)
# =============================================================================
print("\n[09] Train vs Test R²...")

perf_l = pd.read_csv(os.path.join(NB5, 'model_performance_level.csv'))
perf_d = pd.read_csv(os.path.join(NB5, 'model_performance_delta.csv'))
perf_l = perf_l[perf_l['Model'] != 'XGBoost'].reset_index(drop=True)
perf_d = perf_d[perf_d['Model'] != 'XGBoost'].reset_index(drop=True)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14)

for col_idx, (perf, panel) in enumerate([(perf_l, 'ECI Level'), (perf_d, 'ΔECI')], 1):
    models   = perf['Model'].tolist()
    train_r2 = perf['Train R²'].tolist()
    test_r2  = perf['Test R²'].tolist()

    for m, tr, te in zip(models, train_r2, test_r2):
        fig.add_trace(go.Scatter(
            x=[tr, te], y=[m, m], mode='lines',
            line=dict(color='#c0c8d4', width=2.5),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=train_r2, y=models, mode='markers',
        marker=dict(symbol='circle', size=13, color=PALETTE['blue'],
                    line=dict(color='white', width=1.5)),
        name='Train R²', showlegend=(col_idx == 1), legendgroup='train',
        hovertemplate='%{y} Train: %{x:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=test_r2, y=models, mode='markers',
        marker=dict(symbol='diamond', size=13, color=PALETTE['red'],
                    line=dict(color='white', width=1.5)),
        name='Test R²', showlegend=(col_idx == 1), legendgroup='test',
        hovertemplate='%{y} Test: %{x:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    fig.update_xaxes(title_text='R²', gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
    fig.update_yaxes(tickfont=dict(size=11), row=1, col=col_idx)

for x_paper, label in [(0.23, 'ECI Level'), (0.77, 'ΔECI')]:
    fig.add_annotation(
        x=x_paper, y=1.04, xref='paper', yref='paper',
        text=f'<b>{label}</b>', showarrow=False,
        font=dict(size=12, color=NAVY, family=FONT),
        xanchor='center', yanchor='bottom',
    )

fig.update_layout(**base_layout(
    height=440, margin=dict(l=130, r=60, t=80, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.06,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig, '09_ml__train_vs_test_r2_all_models', OUT)


# =============================================================================
# CHART 10 — Actual vs Predicted (ECI level + ΔECI side by side)
# =============================================================================
print("\n[10] Actual vs Predicted — ECI + ΔECI...")

preds = pd.read_csv(os.path.join(NB5, 'test_predictions.csv'))

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
)

for col_idx, (actual_col, pred_col, label) in enumerate([
    ('Actual_ECI',   'Predicted_ECI',   'ECI'),
    ('Actual_Delta', 'Predicted_Delta', 'ΔECI'),
], 1):
    actual = preds[actual_col].dropna().values
    pred   = preds.loc[preds[actual_col].notna(), pred_col].values
    codes  = preds.loc[preds[actual_col].notna(), 'Country Code'].values
    names  = preds.loc[preds[actual_col].notna(), 'Country Name'].values

    lims = [min(actual.min(), pred.min()) - 0.1, max(actual.max(), pred.max()) + 0.1]
    mid  = 0.0

    for x0, x1, y0, y1, fc in [
        (lims[0], mid,     lims[0], mid,     'rgba(46,125,74,0.07)'),
        (mid,     lims[1], mid,     lims[1], 'rgba(46,125,74,0.07)'),
        (lims[0], mid,     mid,     lims[1], 'rgba(194,58,58,0.07)'),
        (mid,     lims[1], lims[0], mid,     'rgba(194,58,58,0.07)'),
    ]:
        fig.add_shape(type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
                      fillcolor=fc, line=dict(width=0), layer='below',
                      row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=[lims[0], lims[1]], y=[lims[0], lims[1]],
        mode='lines', line=dict(color=PALETTE['red'], width=1.5, dash='dash'),
        name='45° line', showlegend=(col_idx == 1), legendgroup='line45',
    ), row=1, col=col_idx)

    resid   = np.abs(actual - pred)
    top_idx = set(np.argsort(resid)[::-1][:5])
    mask_n  = np.array([i not in top_idx for i in range(len(actual))])

    fig.add_trace(go.Scatter(
        x=actual[mask_n], y=pred[mask_n], mode='markers',
        marker=dict(size=6, color=PALETTE['blue'], opacity=0.65,
                    line=dict(color='white', width=0.5)),
        name='Test obs.', showlegend=(col_idx == 1), legendgroup='obs',
        customdata=np.stack([codes[mask_n], names[mask_n]], axis=1),
        hovertemplate='<b>%{customdata[1]}</b><br>'
                      'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    out_idx = list(top_idx)
    fig.add_trace(go.Scatter(
        x=actual[out_idx], y=pred[out_idx], mode='markers+text',
        marker=dict(size=9, color=PALETTE['orange'], opacity=0.9,
                    line=dict(color='white', width=1)),
        text=codes[out_idx], textposition='top center', textfont=dict(size=9),
        name='Largest residuals', showlegend=(col_idx == 1), legendgroup='outliers',
        customdata=np.stack([codes[out_idx], names[out_idx]], axis=1),
        hovertemplate='<b>%{customdata[1]}</b><br>'
                      'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    fig.add_hline(y=0, line=dict(color=GRID, width=1), row=1, col=col_idx)
    fig.add_vline(x=0, line=dict(color=GRID, width=1), row=1, col=col_idx)

    fig.update_xaxes(title_text=f'Actual {label} (test set)', range=lims,
                     gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
    fig.update_yaxes(title_text=f'Predicted {label}', range=lims,
                     gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

fig.update_layout(**base_layout(
    height=560, margin=dict(l=70, r=50, t=70, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.04,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig, '10_ml__actual_vs_predicted_eci_test_set', OUT)


# =============================================================================
# CHART 11 — ECI Forecast: two panels (best performers | worst performers)
#            Each panel shows all 54 grey + case studies + top3 or bottom3
# =============================================================================
print("\n[11] ECI Forecast — split into best / worst panels...")

fc   = pd.read_csv(os.path.join(NB5, 'ECI_Forecast_2020_2030.csv'))
rank = pd.read_csv(os.path.join(NB5, 'Country_Ranking_2020_2030.csv'))
perf = pd.read_csv(os.path.join(NB5, 'model_performance_level.csv'))
perf = perf[perf['Model'] != 'XGBoost']
best_rmse = perf.iloc[0]['Test RMSE'] if 'Test RMSE' in perf.columns else 0.08

master = load_master()
hist   = master[['Country Code', 'Country Name', 'Year',
                  'Economic Complexity Index']].dropna()

rank_sorted  = rank.sort_values('Total_Change', ascending=False).reset_index(drop=True)
CASE_STUDIES = ['COG', 'AZE', 'CHL']
top3    = [cc for cc in rank_sorted['Country Code'].tolist() if cc not in CASE_STUDIES][:3]
bottom3 = [cc for cc in rank_sorted['Country Code'].tolist()[::-1] if cc not in CASE_STUDIES][:3]

CASE_COL = '#4a6fa5'
TOP_COL  = '#2e7d4a'
BOT_COL  = '#c23a3a'
GREY     = '#b0b8c4'

all_eci = pd.concat([
    hist[hist['Country Code'].isin(INCLUDE_LIST)]['Economic Complexity Index'],
    fc[fc['Country Code'].isin(INCLUDE_LIST)]['Ensemble'],
]).dropna()
Y_RANGE = [all_eci.min() - 0.15, all_eci.max() + 0.15]

def _add_country_traces(fig, cc, cname, col, lw, opacity, legendgroup,
                        row_n, col_n, show_legend=False):
    """Draw historical solid + forecast dashed (+ confidence band for highlights)."""
    h = hist[hist['Country Code'] == cc].sort_values('Year')
    f = fc[fc['Country Code'] == cc].sort_values('Year')
    if h.empty or f.empty:
        return None
    ens = f['Ensemble'].values
    yrs = f['Year'].values

    fig.add_trace(go.Scatter(
        x=h['Year'], y=h['Economic Complexity Index'],
        mode='lines', line=dict(color=col, width=lw), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Historical</extra>',
    ), row=row_n, col=col_n)

    if col != GREY:
        rgb = ','.join(str(v) for v in _hex_to_rgb(col))
        fig.add_trace(go.Scatter(
            x=np.concatenate([yrs, yrs[::-1]]).tolist(),
            y=np.concatenate([ens + best_rmse, (ens - best_rmse)[::-1]]).tolist(),
            fill='toself', fillcolor=f'rgba({rgb},0.08)',
            line=dict(color='rgba(0,0,0,0)'),
            showlegend=False, hoverinfo='skip', legendgroup=legendgroup,
        ), row=row_n, col=col_n)

    last_yr  = int(h['Year'].iloc[-1])
    last_eci = float(h['Economic Complexity Index'].iloc[-1])
    fig.add_trace(go.Scatter(
        x=[last_yr] + yrs.tolist(), y=[last_eci] + ens.tolist(),
        mode='lines', line=dict(color=col, width=lw, dash='dash'), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Forecast</extra>',
    ), row=row_n, col=col_n)
    return float(ens[-1])


def _deconflict(label_info, min_gap=0.22):
    sorted_lbl = sorted(label_info.items(), key=lambda x: x[1][0])
    adjusted   = {}
    floor_y    = None
    for cc, (y_act, _) in sorted_lbl:
        y_place = y_act if floor_y is None else max(y_act, floor_y + min_gap)
        adjusted[cc] = y_place
        floor_y = y_place
    return adjusted


def _add_labels(fig, label_info, adjusted_y, x_anchor=2031):
    for cc, (y_act, col) in label_info.items():
        fig.add_annotation(
            x=2030, y=y_act,
            ax=x_anchor, ay=adjusted_y[cc],
            axref='x', ayref='y',
            text=f'<b>{cc}</b>',
            showarrow=True,
            arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
            font=dict(size=9.5, color=col, family=FONT),
            xanchor='left', yanchor='middle',
        )


fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.07,
)

for panel_col, highlight_group, highlight_col, grp_name in [
    (1, top3,    TOP_COL, 'top3'),
    (2, bottom3, BOT_COL, 'bottom3'),
]:
    # shared x-axis decorations
    fig.add_vrect(x0=2019.5, x1=2030.5, fillcolor='rgba(200,210,225,0.18)',
                  line=dict(width=0), layer='below', row=1, col=panel_col)
    fig.add_vline(x=2019.5, line=dict(color='#aaa', width=1.5, dash='dot'),
                  row=1, col=panel_col)

    highlighted_here = set(CASE_STUDIES + highlight_group)

    # 1. Grey background — all non-highlighted
    for cc in INCLUDE_LIST:
        if cc in highlighted_here:
            continue
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        _add_country_traces(fig, cc, cname_, GREY, lw=0.7, opacity=0.3,
                            legendgroup='others', row_n=1, col_n=panel_col)

    label_info = {}

    # 2. Highlight group (top3 or bottom3)
    for cc in highlight_group:
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        y_end  = _add_country_traces(fig, cc, cname_, highlight_col, lw=2.2, opacity=1.0,
                                     legendgroup=grp_name, row_n=1, col_n=panel_col)
        if y_end is not None:
            label_info[cc] = (y_end, highlight_col)

    # 3. Case studies
    for cc in CASE_STUDIES:
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        y_end  = _add_country_traces(fig, cc, cname_, CASE_COL, lw=2.5, opacity=1.0,
                                     legendgroup='cases', row_n=1, col_n=panel_col)
        if y_end is not None:
            label_info[cc] = (y_end, CASE_COL)

    adjusted_y = _deconflict(label_info)

    # Annotations use panel-specific xref/yref
    xref = 'x' if panel_col == 1 else 'x2'
    yref = 'y' if panel_col == 1 else 'y2'
    for cc, (y_act, col) in label_info.items():
        fig.add_annotation(
            x=2030, y=y_act,
            ax=2031, ay=adjusted_y[cc],
            axref=xref, ayref=yref,
            xref=xref, yref=yref,
            text=f'<b>{cc}</b>',
            showarrow=True,
            arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
            font=dict(size=9.5, color=col, family=FONT),
            xanchor='left', yanchor='middle',
        )

    fig.update_xaxes(title_text='Year', gridcolor=GRID, gridwidth=0.5,
                     dtick=5, range=[1994, 2033], row=1, col=panel_col)
    fig.update_yaxes(title_text='Economic Complexity Index' if panel_col == 1 else '',
                     range=Y_RANGE, gridcolor=GRID, gridwidth=0.5, row=1, col=panel_col)

# Shared legend entries
for lbl, col, rk, grp in [
    ('Case studies — COG · AZE · CHL',       CASE_COL, 1, 'cases'),
    ('Top 3 improvers — GNQ · MNG · ECU',    TOP_COL,  2, 'top3'),
    ('Bottom 3 decliners — ZWE · SAU · KAZ', BOT_COL,  3, 'bottom3'),
    ('Other countries',                       GREY,     4, 'others'),
]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col, width=2.5), name=lbl,
        legendgroup=grp, showlegend=True, legendrank=rk))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5), name='── Historical',
    showlegend=True, legendrank=10))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5, dash='dash'), name='- - Forecast',
    showlegend=True, legendrank=11))

fig.update_layout(**base_layout(
    height=620, margin=dict(l=70, r=20, t=70, b=110),
    legend=dict(
        orientation='h', font=dict(size=9.5),
        bgcolor='rgba(255,255,255,0.92)', bordercolor=GRID, borderwidth=1,
        x=0.0, y=-0.15, xanchor='left', yanchor='top', tracegroupgap=0,
    ),
))
save(fig, '11_ml__eci_forecast_top_improvers_2020_2030', OUT)


# =============================================================================

# =============================================================================
# CHART 13 — ECI Forecast Heatmap (generated in Section 2 above)
# =============================================================================
# Chart 13 was already saved during the NB5 training section (Section 2 above).
# It is saved to Final/charts/13_ml__eci_forecast_heatmap_all_54_countries.html
print("\n[13] Chart 13 (Forecast Heatmap) was saved during Section 2 model training.")

## Section 4 — Regression Analysis (NB6): Charts 14–17, 30

Runs the full NB6 econometric pipeline:
- Data loading and variable preparation
- Model 1 (kitchen-sink pooled OLS)
- Model 2 (AR baseline)
- Models 3a/3b (interaction models, Driscoll-Kraay SEs)
- Full-sample comparison

**Charts produced:**
- **14**: ECI distribution shift 1995 vs 2019
- **16**: Coefficient comparison Model 3a vs 3b
- **30**: Regression variable correlation heatmap

Charts 15 and 17 are produced in Section 5 (with improved styling from viz_updates).

In [ ]:
# NB6 uses OUT for 'Final/NB6' intermediary outputs
# Overriding NB5's OUT variable for this section
import os
OUT = 'Final/NB6'
os.makedirs(OUT, exist_ok=True)


# # Descriptive Statistics & Regressions — Economic Complexity and Natural Resources
# 
# **Moody's Ratings Capstone — Industrial Upgrading in Emerging Markets**
# **NB6 of 6 · Pipeline Step: Econometric Analysis (Unified)**
# 
# Dependent variable: Economic Complexity Index (ECI, raw), 1995–2019.
# Sample: 54 resource-dependent developing countries (NR rents ≥ 5 % of GDP in 1995).
# Inputs: `intermediary/Master.csv`, `intermediary/clusters1995.csv` (from NB3 and NB4).
# 
# ---
# 
# ### Changes from the previous regression notebook
# 
# This notebook consolidates and revises the earlier regression analysis in
# `Descriptive_Statistics_and_Regressions_(clean_code).ipynb`. Changes are methodological,
# not merely organisational.
# 
# **1. Dependent variable: raw ECI throughout.**
# The old notebook uses `log_ECI = log(ECI - min + 1)` for the kitchen-sink and AR models,
# then switches to raw ECI levels for the interaction specifications. The shift-and-log
# transformation compresses the distribution and produces coefficients with no direct
# interpretation. NB6 uses raw ECI in every model.
# 
# **2. Lagged predictor in AR model corrected.**
# The old AR baseline regresses `log_ECI` on `log_ECI_lag1` — a lagged log of the shifted
# variable. NB6 regresses raw ECI on `ECI_lag1`. This is consistent with change 1 and
# removes the implicit unit-change interpretation problem from the lag coefficient.
# 
# **3. Kitchen-sink SE clustering corrected: K-means groups → countries.**
# The old Model 1 passes `groups=Cluster` (4 K-means IDs) to the clustered sandwich
# estimator. With only 4 groups the estimator is degenerate regardless of any small-sample
# correction. NB6 clusters by `Country Code` (54 groups). The kitchen-sink is still
# over-parameterised and should not be used for inference, but at least the clustering
# dimension is meaningful.
# 
# **4. Standard errors for core models: Driscoll-Kraay replaces country-clustered.**
# The old notebook uses `cov_type='cluster'` (country-level) in all regressions. This
# corrects serial correlation within countries but not cross-sectional dependence from
# common shocks (commodity price cycles). Models 3a and 3b use Driscoll-Kraay SEs
# (`HAC-Groupsum`, Bartlett kernel, bandwidth = 2 = floor(25^0.25)). Models 1 and 2
# retain clustered SEs. A DK-vs-clustered comparison toggle (`SHOW_CLUSTERED_SE_COMPARISON`)
# is included for robustness checks.
# 
# **5. Systematic mean-centring of interaction terms.**
# The old notebook centres logged variables before computing interactions only in the
# "no log variables" specification, not in the lagged-all-variables model. NB6 applies
# grand-mean centring consistently in every interaction model, so main-effect coefficients
# are always interpretable at the sample mean of the interacted variable.
# 
# **6. Residual diagnostics added.**
# NB6 adds QQ plots and Breusch-Pagan tests for Models 3a and 3b. DK SEs are robust to
# heteroskedasticity and serial correlation, so these diagnostics do not invalidate
# inference but are reported for econometric credibility.
# 
# **7. Full-sample comparison added.**
# NB6 re-estimates Models 3a and 3b on the full `Master.csv` (all available countries)
# and compares coefficients. Interaction terms are mean-centred separately for each
# sample; magnitudes are not directly comparable but sign and significance are.
# 
# **8. `Landlocked` removed from Model 1.**
# Time-invariant in pooled OLS without country fixed effects, so its coefficient
# conflates geography with all unobserved between-country differences.
# 
# **9. Dropped exploratory specifications.**
# 
# - *All-variables-lagged*: lags every structural regressor by one year, halving the
#   usable sample with no theoretical justification for uniform one-year delays. Lagged
#   ECI alone is retained as a persistence control (Model 3b).
# - *Resource-type dummies* (`Hydrocarbons_Dominant`, `Subsoil_Metals_Dominant`,
#   `Precious_Metals_Dominant`): collinear with per-capita production value; the
#   interaction term already captures how resource type modifies the complexity return.
# - *Delta-ECI regression*: equivalent to Model 3b with the lag coefficient constrained
#   to 1. Model 3b estimates that coefficient freely.
# 
# **10. Descriptive statistics section restructured.**
# The old notebook included a thematic variable grouping table, an ECI mean/IQR
# time-series chart (noted inline as "not useful because ECI is normalised each year"),
# a correlation heatmap grouped by theme, resource-type summary stats, and a top-10
# winners/losers table. NB6 replaces these with four focused charts tied directly to the
# regression: ECI distribution shift (1995 vs 2019), median ECI trajectory by cluster,
# correlation matrix for regression variables, and the HCI-production quartile scatter.
# The winners/losers table and resource-type summary are dropped.
# 
# **11. HCI-ECI median-split chart replaced.**
# The old notebook fit separate linear slopes for above/below-median production per
# capita groups and exported two static charts. NB6 replaces this with a
# production-quartile-coloured scatter (section 7b), which conveys the same interaction
# hypothesis without an arbitrary median-split model.
# 
# **12. Model consolidation and portable paths.**
# The old notebook had approximately eight regression cells with duplicated variable
# lists and data-cleaning steps, and hardcoded Windows paths
# (`C:/Users/emili/OneDrive/...`). NB6 consolidates to four numbered models using shared
# helper functions and reads from relative paths under `intermediary/`.
# 
# ---
# 
# ### Specification summary
# 
# | Model | Regressors | SE type | Purpose |
# |---|---|---|---|
# | **Model 1** | 44 controls (kitchen-sink) | Clustered (country) | Sign-checking only |
# | **Model 2** | Lagged ECI only | Clustered (country) | Persistence benchmark |
# | **Model 3a** | 6 vars + 2 interactions | Driscoll-Kraay | Core specification (no lag) |
# | **Model 3b** | 6 vars + 2 interactions + lagged ECI | Driscoll-Kraay | Core specification (with lag) |

import os
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.stattools import durbin_watson
import scipy.stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from types import SimpleNamespace
warnings.filterwarnings('ignore')

# ── Output directory ─────────────────────────────────────────────────────────
OUT = os.path.join('Final', 'NB6')
os.makedirs(OUT, exist_ok=True)

# ── Shared style (matches NB5) ───────────────────────────────────────────────
STYLE = {
    'font_family':      'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size':        11,
    'axis_title_size':  13,
    'legend_size':      11,
    'annotation_size':  12,
    'title_color':      '#1a2744',
    'template':         'plotly_white',
    'plot_bg':          '#fafafa',
    'paper_bg':         '#fafafa',
    'chart_height':     550,
    'chart_height_tall':700,
    'margin':           dict(l=60, r=40, t=10, b=50),
    'margin_bar':       dict(l=220, r=100, t=10, b=50),
    'grid_color':       '#e5e7eb',
    'grid_width':       0.5,
    'zero_line_color':  '#c9cfd6',
}

PALETTE = {
    'blue':        '#4a6fa5',
    'red':         '#c23a3a',
    'green':       '#2e7d4a',
    'orange':      '#d4853b',
    'light_blue':  '#7a9dc4',
    'light_red':   '#d46b6b',
    'light_green': '#5aa87a',
    'dark':        '#3d4f5f',
    'grey':        '#999999',
    'gold':        '#e6b980',
}

CLUSTER_COLORS = ['#4a6fa5', '#c23a3a', '#2e7d4a', '#d4853b']

WRITE_CONFIG = {'displayModeBar': False, 'responsive': True}

plt.rcParams.update({
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
})


def base_layout(**kwargs) -> dict:
    layout = dict(
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        font=dict(family=STYLE['font_family'], size=STYLE['tick_size'],
                  color=STYLE['title_color']),
        margin=STYLE['margin'],
        height=STYLE['chart_height'],
    )
    layout.update(kwargs)
    return layout


def save_chart(fig, path_no_ext: str, width: int = 1100, height: int = 600):
    """No-op: chart output is handled by viz_updates.py / Final/charts/."""
    pass

# ## 0. Data Loading
# 
# The master panel was assembled in NB1-NB3: NB1 merged raw sources (World Bank WDI,
# Penn World Table, V-Dem, Energy Institute Statistical Review, USGS mineral data),
# NB2 diagnosed missingness patterns, and NB3 imputed gaps using three-year linear
# interpolation (capped) followed by KNN for remaining holes.
# 
# The 54-country sample is defined by a **baseline threshold**: total natural resource
# rents ≥ 5 % of GDP in the year 1995. The threshold is applied only at the baseline to
# avoid endogeneity. If it were applied at every time period, countries that successfully
# diversified away from resources would exit the sample, biasing estimates.
# 
# Cluster assignments come from NB4, where PCA on per-capita log-transformed production
# data was followed by K-means clustering (k = 4). These clusters group countries by
# resource profile (broadly: hydrocarbon-dominated, subsoil-metals, mixed mineral,
# low-production) and are used only for descriptive purposes in this notebook.

# ── Load pipeline outputs ─────────────────────────────────────────────────────
master       = pd.read_csv('intermediary/Master.csv')
cluster_1995 = pd.read_csv('intermediary/clusters1995.csv')

print(f"Master:   {len(master):,} obs, {master['Country Code'].nunique()} countries, "
      f"{master['Year'].nunique()} years ({int(master['Year'].min())}–{int(master['Year'].max())})")
print(f"Clusters: {len(cluster_1995)} country assignments")

# ── 54-country sample (same include_list as NB4) ──────────────────────────────
INCLUDE = [
    'AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
    'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
    'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
    'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
    'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
    'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE',
]

df = master[master['Country Code'].isin(INCLUDE)].copy()

# ── Per-capita production/reserves values ─────────────────────────────────────
df['Total_Production_Value_Per_Capita'] = df['Total_Production_Value'] / df['Population']
df['Total_Reserves_Value_Per_Capita']   = df['Total_Reserves_Value']   / df['Population']

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# ── Merge cluster labels (1995 assignments, time-invariant) ───────────────────
cluster_1995 = cluster_1995[['Country Code', 'Cluster']]
df = df.merge(cluster_1995, on='Country Code', how='left')

print(f"\nSample: {df['Country Code'].nunique()} countries × {df['Year'].nunique()} years "
      f"= {len(df):,} obs")
print(f"ECI range: {df['Economic Complexity Index'].min():.3f} – "
      f"{df['Economic Complexity Index'].max():.3f}")

df.to_csv('intermediary/high_resource_countries.csv', index=False)
print(f"  ✓ intermediary/high_resource_countries.csv")

# ## 1. Descriptive Statistics
# 
# Summary statistics for key variables across the 54-country sample.

# ── Key variables for descriptive table ──────────────────────────────────────
DESC_VARS = {
    'Economic Complexity Index': 'ECI',
    'Human capital index':                          'Human capital index',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'GFCF (% GDP)',
    'Domestic credit to private sector (% of GDP)': 'Domestic credit (% GDP)',
    'Access to electricity (% of population)':      'Electricity access (%)',
    'Rule of law index':                            'Rule of law',
    'Political stability — estimate':               'Political stability',
    'Total natural resources rents (% of GDP)':     'NR rents (% GDP)',
    'Total_Production_Value_Per_Capita':            'Prod. value per capita (USD)',
    'Trade (% of GDP)':                             'Trade (% GDP)',
}

desc = df[list(DESC_VARS.keys())].rename(columns=DESC_VARS)
print(desc.describe().round(3).to_string())
print(f"\nN obs: {len(df):,}  |  N countries: {df['Country Code'].nunique()}")

# ### 1a. ECI Distribution Change: 1995 vs 2019

yr_95 = df[df['Year'] == 1995]['Economic Complexity Index'].dropna().sort_values().values
yr_19 = df[df['Year'] == 2019]['Economic Complexity Index'].dropna().sort_values().values

fig = go.Figure()
for vals, yr, col in [(yr_95, 1995, PALETTE['blue']), (yr_19, 2019, PALETTE['red'])]:
    pcts = np.linspace(0, 100, len(vals))
    fig.add_trace(go.Scatter(
        x=pcts, y=vals,
        mode='lines', name=str(yr),
        line=dict(color=col, width=2.5),
    ))

fig.update_layout(**base_layout(
    height=STYLE['chart_height'],
    xaxis=dict(
        title=dict(text='Percentile', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
    ),
    yaxis=dict(
        title=dict(text='Economic Complexity Index', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        zeroline=True, zerolinecolor=STYLE['zero_line_color'], zerolinewidth=1,
    ),
    legend=dict(font=dict(size=STYLE['legend_size'])),
))

os.makedirs('Final/charts', exist_ok=True)
fig.write_html(os.path.join('Final/charts', '14_reg__eci_distribution_shift_1995_vs_2019.html'), config=WRITE_CONFIG)
print('  Saved: 14_reg__eci_distribution_shift_1995_vs_2019.html')

# ### 1b. Median ECI Trajectory by Resource-Profile Cluster

traj = (df.groupby(['Year', 'Cluster'])['Economic Complexity Index']
          .median().reset_index())

clusters_present = sorted(traj['Cluster'].dropna().unique())

fig = go.Figure()
for i, cl in enumerate(clusters_present):
    sub = traj[traj['Cluster'] == cl]
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers', name=str(cl),
        line=dict(color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)], width=2),
        marker=dict(size=5),
    ))

fig.update_layout(**base_layout(
    xaxis=dict(
        title=dict(text='Year', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        dtick=5,
    ),
    yaxis=dict(
        title=dict(text='Median ECI', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        zeroline=True, zerolinecolor=STYLE['zero_line_color'], zerolinewidth=1,
    ),
    legend=dict(title=dict(text='Cluster'), font=dict(size=STYLE['legend_size'])),
))

# Chart 15 is regenerated by the viz_updates section below with better styling

# ### 1c. Correlation Matrix — Key Variables

corr_cols = list(DESC_VARS.keys())
corr_df   = df[corr_cols].rename(columns=DESC_VARS).corr().round(2)

labels = list(corr_df.columns)
z      = corr_df.values

fig = go.Figure(go.Heatmap(
    z=z, x=labels, y=labels,
    colorscale=[
        [0.0, PALETTE['red']], [0.5, '#fafafa'], [1.0, PALETTE['blue']]
    ],
    zmid=0, zmin=-1, zmax=1,
    text=z.round(2), texttemplate='%{text}',
    textfont=dict(size=9, family=STYLE['font_family']),
    hovertemplate='%{x} × %{y}: %{z:.2f}<extra></extra>',
    colorbar=dict(thickness=14, len=0.9,
                  tickfont=dict(size=STYLE['tick_size'],
                                family=STYLE['font_family'])),
))

fig.update_layout(**base_layout(
    height=620,
    margin=dict(l=180, r=60, t=10, b=180),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
))

fig.write_html(os.path.join('Final/charts', '30_diag__regression_variable_correlation_heatmap.html'), config=WRITE_CONFIG)
print('  Saved: 30_diag__regression_variable_correlation_heatmap.html')

# ## 2. Regression Setup
# 
# Four OLS specifications are estimated. All use **raw ECI** as the dependent variable.
# None include country or year fixed effects; the pooled design captures both
# between-country and within-country variation.
# 
# **Standard errors.**
# Models 1 and 2 use clustered standard errors at the country level (54 groups).
# Models 3a and 3b use **Driscoll-Kraay** standard errors (`HAC-Groupsum`, Bartlett
# kernel, bandwidth = 2), which are robust to both within-country serial correlation
# and cross-country dependence within the same year.
# 
# **On including lagged ECI.**
# Model 3b adds last year's ECI as a right-hand-side variable. This controls for the
# strong persistence in economic complexity: because ECI barely moves from year to year,
# the remaining coefficients in 3b reflect associations with the *change* in ECI
# conditional on last year's level. If the interaction coefficients are stable between
# 3a and 3b, they capture structural relationships rather than proxying for omitted
# dynamic factors.
# 
# Note that this approach is more flexible than regressing the year-on-year change
# (ΔECI = ECI_t minus ECI_{t-1}) directly on the regressors. The delta-ECI approach
# implicitly constrains the lagged-ECI coefficient to equal exactly 1. Model 3b
# estimates that coefficient freely from the data.

# ── Full variable list (Model 1) ──────────────────────────────────────────────
# Note: 'Landlocked' removed. It is time-invariant and conflates geography with
# other between-country variation in pooled OLS.
INDEP_VARS = [
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'Agriculture',
    'Capital depreciation rate',
    'Clientelism index',
    'Death rates, crude per 1000 people',
    'Domestic credit to private sector (% of GDP)',
    'GDP per capita (constant prices, PPP)',
    'Government revenue',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Human capital index',
    'Industry',
    'Inflation, consumer prices (annual %)',
    'Lending interest rate (%)',
    'Life expectancy at birth, total (years)',
    'Manufacturing',
    'Mineral rents (% of GDP)',
    'Mobile cellular subscriptions (per 100 people)',
    'Natural gas rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Political corruption index',
    'Political stability — estimate',
    'Primary net lending, General government, Percent of GDP',
    'Property rights',
    'Real interest rate (%)',
    'Rule of law index',
    'Services',
    'Share of consumption in GDP',
    'Share of government spending in GDP',
    'Share of investment in GDP',
    'Total natural resources rents (% of GDP)',
    'Trade (% of GDP)',
    'Urban population (% of total population)',
    'Use of IMF credit (DOD, current US$)',
    'Total_Production',
    'Total_Reserves',
    'Total_Production_Value',
    'Total_Reserves_Value',
    'Total_Production_Value_Per_Capita',
    'Total_Reserves_Value_Per_Capita',
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
    'Population',
]

# ── Parsimonious variable list (Models 3a, 3b) ───────────────────────────────
# Selected on theoretical grounds from the resource-curse literature:
#   - Human capital and institutional quality: most-cited channels for
#     resource wealth affecting diversification (Gylfason 2001, Mehlum et al. 2006)
#   - Per-capita production value: resource extraction intensity relative to
#     population size, closer to the standard resource-dependence measure
#   - Trade openness: exposure to international competition and learning
PARSIMONIOUS_VARS = [
    'Human capital index',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Political stability — estimate',
    'Rule of law index',
    'Total_Production_Value_Per_Capita',
    'Trade (% of GDP)',
]

# ── Log transforms ────────────────────────────────────────────────────────────
# log1p(x) = ln(1 + x) handles zeros safely (ln(0) is undefined).
# Applied to HCI, GFCF, and per-capita production value because these span
# several orders of magnitude; the log reduces outlier influence and allows
# coefficients to be interpreted in terms of proportional changes.
df['log_HCI']              = np.log1p(df['Human capital index'])
df['log_GFCF']             = np.log1p(df['Gross fixed capital formation, all, Constant prices, Percent of GDP'])
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'])

# ── Mean-centred logs for interactions ────────────────────────────────────────
# Centring subtracts the grand (sample-wide) mean from each log-transformed
# variable before computing the product. This reduces multicollinearity between
# the interaction term and its constituent main effects. After centring, the
# main-effect coefficients are interpretable as the association at the sample
# mean of the interacted variable.
for col in ['log_HCI', 'log_GFCF', 'log_Production_Value']:
    df[f'{col}_c'] = df[col] - df[col].mean()

df['log_HCI_x_log_Production']  = df['log_HCI_c']  * df['log_Production_Value_c']
df['log_GFCF_x_log_Production'] = df['log_GFCF_c'] * df['log_Production_Value_c']

# ── Lagged ECI (raw) ─────────────────────────────────────────────────────────
df = df.sort_values(['Country Code', 'Year'])
df['ECI_lag1'] = df.groupby('Country Code')['Economic Complexity Index'].shift(1)

print("Variables ready.")
print(f"  Full set (Model 1):        {len(INDEP_VARS)} vars")
print(f"  Parsimonious (Models 3a/b): {len(PARSIMONIOUS_VARS)} vars + 2 interactions")
print(f"  Lagged ECI available:       {df['ECI_lag1'].notna().sum():,} obs")

# ## 3. Model 1 — Pooled OLS, Full Variable Set
# 
# **Note on over-parameterisation:** With 44 regressors and 54 country clusters, the clustered covariance matrix approaches rank deficiency. Treat this as a sign-checking reference only. The clustered SEs are unreliable in this configuration and should not be used for inference. If reviewers challenge this specification, the defensible response is either to drop it entirely or reduce to a 15–20 variable intermediate specification motivated by the same theoretical channels as Models 3a/3b.
# 
# All 44 controls, clustered standard errors (by country). Sign-checking reference only.
# 

reg1_cols = INDEP_VARS + ['Economic Complexity Index', 'Country Code']
reg1_df   = df[reg1_cols].dropna()

y1 = reg1_df['Economic Complexity Index']
X1 = sm.add_constant(reg1_df[INDEP_VARS])

m1 = sm.OLS(y1, X1).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg1_df['Country Code']},
)

print("=" * 70)
print("MODEL 1 — Pooled OLS, Full Variable Set (Clustered SE by Country)")
print("=" * 70)
print(f"  N obs:        {int(m1.nobs):,}")
print(f"  N countries:  {reg1_df['Country Code'].nunique()}")
print(f"  R²:           {m1.rsquared:.4f}")
print(f"  Adj. R²:      {m1.rsquared_adj:.4f}")
print(f"  Durbin-Watson: {durbin_watson(m1.resid):.3f}")
print()
print(f"{'Variable':<52} {'Coef':>9} {'SE':>9} {'t':>7} {'p':>7}")
print("-" * 90)
for v in [c for c in m1.params.index if c != 'const']:
    coef, se, t, p = m1.params[v], m1.bse[v], m1.tvalues[v], m1.pvalues[v]
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    if p < 0.1:
        print(f"{v:<52} {coef:>9.4f} {se:>9.4f} {t:>7.2f} {p:>7.4f} {sig}")

# ## 4. Model 2 — AR Baseline (Lagged ECI)
# 
# Autoregressive baseline: only lagged ECI as predictor. The high R² (typically ~0.97)
# reflects the strong persistence of economic complexity. A country's ECI today is
# almost entirely determined by its ECI last year. This baseline matters because any
# model claiming to explain ECI must be benchmarked against this inherent persistence.

reg2_cols = ['Economic Complexity Index', 'ECI_lag1', 'Country Code']
reg2_df   = df[reg2_cols].dropna()

y2 = reg2_df['Economic Complexity Index']
X2 = sm.add_constant(reg2_df[['ECI_lag1']])

m2 = sm.OLS(y2, X2).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg2_df['Country Code']},
)

print("=" * 70)
print("MODEL 2 — AR Baseline: Lagged ECI (Clustered SE by Country)")
print("=" * 70)
print(f"  N obs:        {int(m2.nobs):,}")
print(f"  N countries:  {reg2_df['Country Code'].nunique()}")
print(f"  R²:           {m2.rsquared:.4f}")
print(f"  Adj. R²:      {m2.rsquared_adj:.4f}")
print(f"  Durbin-Watson: {durbin_watson(m2.resid):.3f}")
print()
for v in m2.params.index:
    coef, se, t, p = m2.params[v], m2.bse[v], m2.tvalues[v], m2.pvalues[v]
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    print(f"  {v:<30} coef={coef:>8.4f}  SE={se:>8.4f}  t={t:>7.2f}  p={p:>7.4f} {sig}")

# ## 5. Model 3 — Interaction Model (HCI × Production, GFCF × Production)
# 
# Parsimonious specification with two interaction terms, estimated with and without
# lagged ECI as a control. Variables are mean-centred (grand mean) before computing
# interactions to reduce multicollinearity.
# 
# **Model 3a**: without lagged ECI. Captures the total cross-sectional and longitudinal
# association between the regressors and ECI.
# 
# **Model 3b**: with lagged ECI. Coefficients now reflect associations with ECI
# conditional on last year's level. If the interaction coefficients are stable across
# 3a and 3b, they capture structural relationships rather than proxying for omitted
# dynamic factors.
# 
# Driscoll-Kraay standard errors (HAC-Groupsum, Bartlett kernel, bandwidth = 2).

# ── Driscoll-Kraay fitting helper ────────────────────────────────────────────
INTERACT_VARS = ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production']

reg3_input = ['log_HCI', 'log_GFCF', 'Political stability — estimate',
              'Rule of law index', 'log_Production_Value', 'Trade (% of GDP)']


def fit_driscoll_kraay(y, X, time, groups, label=''):
    """Fit OLS with Driscoll-Kraay SEs. Returns (SimpleNamespace, raw_fit)."""
    raw = sm.OLS(y, X).fit()
    robust = raw.get_robustcov_results(
        cov_type='HAC-Groupsum',
        time=time, groups=groups,
        maxlags=2,  # floor(T^(1/4)) = floor(25^0.25) ≈ 2.2 → bw=2 is defensible
        kernel='bartlett', use_correction=True,
    )

    ns = SimpleNamespace(
        params       = pd.Series(robust.params,  index=X.columns),
        bse          = pd.Series(robust.bse,     index=X.columns),
        tvalues      = pd.Series(robust.tvalues, index=X.columns),
        pvalues      = pd.Series(robust.pvalues, index=X.columns),
        nobs         = robust.nobs,
        rsquared     = robust.rsquared,
        rsquared_adj = robust.rsquared_adj,
        cov_params   = robust.cov_params,
    )

    print(f"\n{'=' * 70}")
    print(f"{label}")
    print(f"{'=' * 70}")
    print(f"  N obs:        {int(ns.nobs):,}")
    print(f"  N countries:  {groups.nunique()}")
    print(f"  R²:           {ns.rsquared:.4f}")
    print(f"  Adj. R²:      {ns.rsquared_adj:.4f}")
    print(f"  Durbin-Watson: {durbin_watson(raw.resid):.3f}")
    print()
    print(f"  {'Variable':<43} {'Coef':>9} {'SE':>9} {'t':>7} {'p':>7}")
    print("  " + "-" * 76)
    for v in ns.params.index:
        coef, se, t, p = ns.params[v], ns.bse[v], ns.tvalues[v], ns.pvalues[v]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        print(f"  {v:<43} {coef:>9.4f} {se:>9.4f} {t:>7.2f} {p:>7.4f} {sig}")
    return ns, raw


# ── Model 3a: without lagged ECI ─────────────────────────────────────────────
reg3_cols  = reg3_input + INTERACT_VARS + ['Economic Complexity Index', 'Country Code', 'Year']
reg3_df    = df[reg3_cols].dropna()

y3a = reg3_df['Economic Complexity Index']
X3a = sm.add_constant(reg3_df[reg3_input + INTERACT_VARS])

m3a, m3a_raw = fit_driscoll_kraay(
    y3a, X3a, reg3_df['Year'], reg3_df['Country Code'],
    label='MODEL 3a — Interaction Model, NO lag (Driscoll-Kraay SE)')


# ── Model 3b: with lagged ECI ────────────────────────────────────────────────
reg3b_cols = reg3_cols + ['ECI_lag1']
reg3b_df   = df[reg3b_cols].dropna()

y3b = reg3b_df['Economic Complexity Index']
X3b = sm.add_constant(reg3b_df[reg3_input + INTERACT_VARS + ['ECI_lag1']])

m3b, m3b_raw = fit_driscoll_kraay(
    y3b, X3b, reg3b_df['Year'], reg3b_df['Country Code'],
    label='MODEL 3b — Interaction Model, WITH lag (Driscoll-Kraay SE)')

# ── Quick comparison ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMPARISON: Model 3a vs 3b")
print("=" * 70)
print(f"  {'':30} {'3a (no lag)':>14} {'3b (with lag)':>14}")
print(f"  {'R²':<30} {m3a.rsquared:>14.4f} {m3b.rsquared:>14.4f}")
for v in reg3_input + INTERACT_VARS:
    c_a = m3a.params.get(v, np.nan)
    c_b = m3b.params.get(v, np.nan)
    s_a = '***' if m3a.pvalues.get(v,1)<0.01 else '**' if m3a.pvalues.get(v,1)<0.05 else '*' if m3a.pvalues.get(v,1)<0.1 else ''
    s_b = '***' if m3b.pvalues.get(v,1)<0.01 else '**' if m3b.pvalues.get(v,1)<0.05 else '*' if m3b.pvalues.get(v,1)<0.1 else ''
    print(f"  {v:<30} {c_a:>+10.4f}{s_a:<4} {c_b:>+10.4f}{s_b:<4}")
if 'ECI_lag1' in m3b.params:
    c_lag = m3b.params['ECI_lag1']
    s_lag = '***' if m3b.pvalues['ECI_lag1']<0.01 else '**' if m3b.pvalues['ECI_lag1']<0.05 else '*' if m3b.pvalues['ECI_lag1']<0.1 else ''
    print(f"  {'ECI_lag1':<30} {'':>14} {c_lag:>+10.4f}{s_lag:<4}")

# ── Clustered SE comparison (toggle) ────────────────────────────────────────
# Set SHOW_CLUSTERED_SE_COMPARISON = True to see how results change when
# using simpler country-clustered SEs instead of Driscoll-Kraay.
# DK is preferred (robust to cross-sectional dependence from commodity shocks)
# but reviewers may ask for this comparison to verify robustness.
SHOW_CLUSTERED_SE_COMPARISON = False

if SHOW_CLUSTERED_SE_COMPARISON:
    def _fmt(coef, se, pval):
        s = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        return f'{coef:>+9.4f} ({se:.4f}){s}'

    print('\n' + '=' * 80)
    print('SE ROBUSTNESS: Driscoll-Kraay vs Country-Clustered')
    print('=' * 80)
    for y_r, X_r, df_r, m_dk, spec in [
        (y3a, X3a, reg3_df,  m3a, '3a (no lag)'),
        (y3b, X3b, reg3b_df, m3b, '3b (with lag)'),
    ]:
        m_cl = sm.OLS(y_r, X_r).fit(
            cov_type='cluster', cov_kwds={'groups': df_r['Country Code']}
        )
        print(f'\nModel {spec}')
        print(f"  {'Variable':<40} {'DK SE':>26} {'Clustered SE':>26}")
        print('  ' + '-' * 95)
        for v in [c for c in m_dk.params.index if c != 'const']:
            dk_str  = _fmt(m_dk.params[v], m_dk.bse[v], m_dk.pvalues[v])
            cl_str  = _fmt(m_cl.params[v], m_cl.bse[v], m_cl.pvalues[v])
            print(f'  {v:<40} {dk_str:>26} {cl_str:>26}')


# ── Residual diagnostics — Models 3a / 3b ────────────────────────────────────
# Durbin-Watson is reported in the summary but does not diagnose
# heteroskedasticity or non-normality. Added:
#   1. QQ plot of residuals (visual normality check)
#   2. Breusch-Pagan test (formal heteroskedasticity test)
# Note: Driscoll-Kraay SEs are robust to both heteroskedasticity and serial
# correlation, so finding either does not invalidate the inference — but
# reporting these diagnostics strengthens econometric credibility.
from statsmodels.stats.diagnostic import het_breuschpagan
import scipy.stats as stats

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, m_raw, label in [
    (axes[0], m3a_raw, 'Model 3a (no lag)'),
    (axes[1], m3b_raw, 'Model 3b (with lag)'),
]:
    resid = m_raw.resid
    (osm, osr), (slope, intercept, _) = stats.probplot(resid, dist='norm')
    ax.scatter(osm, osr, s=18, alpha=0.6, color=PALETTE['blue'], edgecolor='none')
    ax.plot(
        [osm[0], osm[-1]],
        [osm[0] * slope + intercept, osm[-1] * slope + intercept],
        color=PALETTE['red'], linewidth=1.5, linestyle='--', label='Normal reference'
    )
    ax.set_xlabel('Theoretical quantiles', fontsize=11)
    ax.set_ylabel('Ordered residuals', fontsize=11)
    ax.set_title(f'Q-Q Plot — {label}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.close("all")

# ── Breusch-Pagan test ────────────────────────────────────────────────────────
print('\nBreusch-Pagan test (H0: homoskedastic):')
for m_raw, X_reg, label in [
    (m3a_raw, X3a, 'Model 3a'),
    (m3b_raw, X3b, 'Model 3b'),
]:
    lm_stat, lm_pval, _, _ = het_breuschpagan(m_raw.resid, X_reg)
    verdict = 'reject H0 — heteroskedastic' if lm_pval < 0.05 else 'fail to reject H0'
    print(f'  {label}: LM={lm_stat:.3f}  p={lm_pval:.4f}  → {verdict}')
print('  (Driscoll-Kraay SEs remain valid regardless of this result)')


# ## 6. Sample Comparison — 54 Resource-Rich vs All Countries
# 
# The 54-country sample was selected based on NR rents ≥ 5 % of GDP in 1995.
# This section re-estimates Models 3a and 3b on the **full Master dataset** (all
# available countries) and compares coefficients. The purpose is to assess whether
# the interaction effects are specific to resource-dependent economies or hold more
# broadly.
# 
# **Note on centring**: the mean-centring of interaction terms is computed separately
# for each sample (using the respective grand mean), so interaction coefficients are
# not directly comparable in magnitude across samples. They are comparable in sign
# and significance.

# ── Prepare the full-sample dataframe ─────────────────────────────────────────
df_all = master.copy()

if 'Unnamed: 0' in df_all.columns:
    df_all = df_all.drop(columns=['Unnamed: 0'])

df_all['Total_Production_Value_Per_Capita'] = df_all['Total_Production_Value'] / df_all['Population']

df_all['log_HCI']              = np.log1p(df_all['Human capital index'])
df_all['log_GFCF']             = np.log1p(df_all['Gross fixed capital formation, all, Constant prices, Percent of GDP'])
df_all['log_Production_Value'] = np.log1p(df_all['Total_Production_Value_Per_Capita'])

# Centre df_all on the 54-country sample means (not the full-sample mean).
# This ensures main-effect coefficients are evaluated at the same point as
# in Models 3a/3b, making the coefficient deltas in the comparison table
# directly interpretable as shifts in effect size, not shifts in evaluation point.
_centre_means = {col: df[col].mean()
                 for col in ['log_HCI', 'log_GFCF', 'log_Production_Value']}
for col in ['log_HCI', 'log_GFCF', 'log_Production_Value']:
    df_all[f'{col}_c'] = df_all[col] - _centre_means[col]

df_all['log_HCI_x_log_Production']  = df_all['log_HCI_c']  * df_all['log_Production_Value_c']
df_all['log_GFCF_x_log_Production'] = df_all['log_GFCF_c'] * df_all['log_Production_Value_c']

df_all = df_all.sort_values(['Country Code', 'Year'])
df_all['ECI_lag1'] = df_all.groupby('Country Code')['Economic Complexity Index'].shift(1)

print(f"Full sample: {df_all['Country Code'].nunique()} countries, "
      f"{df_all['Year'].nunique()} years, {len(df_all):,} obs")


# ── Model 3a (full sample) ───────────────────────────────────────────────────
all_3a_cols = reg3_input + INTERACT_VARS + ['Economic Complexity Index', 'Country Code', 'Year']
all_3a_df   = df_all[all_3a_cols].dropna()

m3a_all, m3a_all_raw = fit_driscoll_kraay(
    all_3a_df['Economic Complexity Index'],
    sm.add_constant(all_3a_df[reg3_input + INTERACT_VARS]),
    all_3a_df['Year'], all_3a_df['Country Code'],
    label='FULL SAMPLE — Model 3a (no lag)')

# ── Model 3b (full sample) ───────────────────────────────────────────────────
all_3b_cols = all_3a_cols + ['ECI_lag1']
all_3b_df   = df_all[all_3b_cols].dropna()

m3b_all, m3b_all_raw = fit_driscoll_kraay(
    all_3b_df['Economic Complexity Index'],
    sm.add_constant(all_3b_df[reg3_input + INTERACT_VARS + ['ECI_lag1']]),
    all_3b_df['Year'], all_3b_df['Country Code'],
    label='FULL SAMPLE — Model 3b (with lag)')


# ── Comparison table ──────────────────────────────────────────────────────────
def sig(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else '   '

core_vars = reg3_input + INTERACT_VARS + ['ECI_lag1']
var_labels = {
    'log_HCI':                       'log(HCI)',
    'log_GFCF':                      'log(GFCF)',
    'Political stability — estimate':'Pol. stability',
    'Rule of law index':             'Rule of law',
    'log_Production_Value':          'log(Prod. val. p.c.)',
    'Trade (% of GDP)':              'Trade (% GDP)',
    'log_HCI_x_log_Production':      'log(HCI) × log(Prod)',
    'log_GFCF_x_log_Production':     'log(GFCF) × log(Prod)',
    'ECI_lag1':                       'Lagged ECI',
}

pairs = [
    ('Model 3a',  m3a,     m3a_all),
    ('Model 3b',  m3b,     m3b_all),
]

print("\n" + "=" * 110)
print("SAMPLE COMPARISON: 54 Resource-Rich Countries vs All Countries")
print("=" * 110)

for spec_name, m_54, m_full in pairs:
    n54   = int(m_54.nobs)
    nfull = int(m_full.nobs)
    nc_full = df_all['Country Code'].nunique()
    print(f"\n{'─' * 110}")
    print(f"  {spec_name:<14} │  54 countries (N={n54:,})           │  All countries (N={nfull:,}, {nc_full} ctries)")
    print(f"  {'':14} │  R² = {m_54.rsquared:.4f}                    │  R² = {m_full.rsquared:.4f}")
    print(f"  {'Variable':<22} │ {'Coef':>9} {'SE':>9} {'':>4}  │ {'Coef':>9} {'SE':>9} {'':>4}  │ Δ coef")
    print(f"  {'─' * 22}─┼{'─' * 27}─┼{'─' * 27}─┼{'─' * 9}")
    for v in core_vars:
        label = var_labels.get(v, v[:22])
        c54  = m_54.params.get(v, None)
        c_all = m_full.params.get(v, None)
        if c54 is None and c_all is None:
            continue
        if c54 is not None:
            se54 = m_54.bse[v]
            s54  = sig(m_54.pvalues[v])
            c54_str = f"{c54:>+9.4f} {se54:>9.4f} {s54}"
        else:
            c54_str = f"{'':>24}"
            c54 = 0
        if c_all is not None:
            se_all = m_full.bse[v]
            s_all  = sig(m_full.pvalues[v])
            c_all_str = f"{c_all:>+9.4f} {se_all:>9.4f} {s_all}"
        else:
            c_all_str = f"{'':>24}"
            c_all = 0
        delta = c_all - c54 if (m_54.params.get(v) is not None and m_full.params.get(v) is not None) else None
        delta_str = f"{delta:>+9.4f}" if delta is not None else ""
        print(f"  {label:<22} │ {c54_str} │ {c_all_str} │ {delta_str}")

print(f"\n{'=' * 110}")
print("Note: 'All countries' = full Master.csv without the NR rents ≥ 5% filter.")
print("Interaction terms use grand-mean centring computed separately for each sample.")

# ## 7. Visualisations

# ### 7a. Coefficient Comparison — Model 3a vs 3b
# 
# Horizontal dot-and-whisker plot comparing point estimates and 95 % confidence
# intervals for the core variables across the two specifications. If a variable's
# confidence interval crosses zero, its association with ECI is not statistically
# significant at the 5 % level.

plot_vars = [v for v in reg3_input + INTERACT_VARS if v != 'const']

labels = [v.replace('log_', 'log(').replace('_x_', ') × log(') + (')' if 'log_' in v else '')
          for v in plot_vars]

models_to_plot = [
    (m3a,  PALETTE['blue'],      'Model 3a (no lag)'),
    (m3b,  PALETTE['light_blue'],'Model 3b (with lag)'),
]

fig = go.Figure()
for model, col, name in models_to_plot:
    coefs  = [model.params.get(v, np.nan)  for v in plot_vars]
    lowers = [model.params.get(v, np.nan) - 1.96 * model.bse.get(v, np.nan) for v in plot_vars]
    uppers = [model.params.get(v, np.nan) + 1.96 * model.bse.get(v, np.nan) for v in plot_vars]
    fig.add_trace(go.Scatter(
        y=labels, x=coefs,
        error_x=dict(
            type='data', symmetric=False,
            array=[u - c for c, u in zip(coefs, uppers)],
            arrayminus=[c - l for c, l in zip(coefs, lowers)],
            color=col, thickness=1.5, width=5,
        ),
        mode='markers',
        marker=dict(color=col, size=8, symbol='circle'),
        name=name,
    ))

fig.add_vline(x=0, line=dict(color=STYLE['zero_line_color'], width=1.5, dash='dash'))

fig.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Coefficient (95 % CI, Driscoll-Kraay)', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        zeroline=False,
    ),
    yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
    legend=dict(font=dict(size=STYLE['legend_size']),
                orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
))

fig.write_html(os.path.join('Final/charts', '16_reg__coefficients_model3a_vs_model3b.html'), config=WRITE_CONFIG)
print('  Saved: 16_reg__coefficients_model3a_vs_model3b.html')

# ### 7b. ECI vs Human Capital — by Production Value Quartile
# 
# Scatter plot of log(HCI) against ECI, with points coloured by per-capita production
# value quartile. If the slope of HCI on ECI steepens at higher production quartiles,
# this provides visual evidence of the positive interaction effect.

plot_df = df[['log_HCI', 'Economic Complexity Index', 'log_Production_Value',
              'Country Code', 'Year']].dropna()

plot_df['Prod_quartile'] = pd.qcut(
    plot_df['log_Production_Value'], q=4,
    labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
)

q_colors = [PALETTE['light_blue'], PALETTE['blue'], PALETTE['orange'], PALETTE['red']]

fig = go.Figure()
for q, col in zip(['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'], q_colors):
    sub = plot_df[plot_df['Prod_quartile'] == q]
    fig.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub['Economic Complexity Index'],
        mode='markers',
        marker=dict(color=col, size=5, opacity=0.65,
                    line=dict(width=0.3, color='white')),
        name=f'Production {q}',
        hovertemplate='%{customdata[0]} %{customdata[1]}<br>'
                      'log(HCI)=%{x:.2f}  ECI=%{y:.2f}<extra></extra>',
        customdata=sub[['Country Code', 'Year']].values,
    ))

fig.update_layout(**base_layout(
    xaxis=dict(
        title=dict(text='log(Human Capital Index)', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
    ),
    yaxis=dict(
        title=dict(text='Economic Complexity Index', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
    ),
    legend=dict(title=dict(text='Prod. Value p.c. Quartile'),
                font=dict(size=STYLE['legend_size'])),
))

# Chart 17 is regenerated by the viz_updates section below with better styling

# ## Summary
# 
# | Item | Detail |
# |------|--------|
# | Sample | 54 countries × 25 years (actual obs vary by model after dropna) |
# | Dependent variable | Raw ECI |
# | Models | Pooled OLS (full), AR baseline, Interaction ×2 (±lag) |
# | SE type (Models 3a/b) | Driscoll-Kraay (HAC-Groupsum, Bartlett kernel, bw=2) |
# | Fixed effects | None (pooled design) |
# | Full-sample comparison | Yes (all countries in Master.csv, Models 3a and 3b) |
# 
# **Key results (expected):**
# 
# - `log(HCI) × log(Production p.c.)`: positive and significant across specifications.
#   Higher human capital amplifies the ECI returns to resource production.
# - `log(GFCF) × log(Production p.c.)`: small, not significant.
# - Adding lagged ECI (Model 3b) absorbs persistence and raises R² substantially.
#   If the other coefficients are stable after adding the lag, that confirms they
#   reflect genuine structural associations rather than proxying for ECI persistence.
# 
# **Methodological notes:**
# - DV is raw ECI (the previous log-of-shifted transformation compressed the
#   distribution and produced hard-to-interpret coefficients).
# - Production value is per-capita (captures resource intensity relative to
#   population rather than absolute scale).
# - No fixed effects: the pooled design retains between-country variation, which
#   is central to the research question.
# - Landlocked dummy removed: conflates geography with other between-country
#   variation in pooled OLS.

print("=" * 70)
print("NB6: REGRESSIONS SUMMARY (Unified)")
print("=" * 70)
print(f"  Sample:         {df['Country Code'].nunique()} countries × {df['Year'].nunique()} years")
print(f"  DV:             Raw ECI")
print()
print(f"  Model          N obs    R²      Adj R²   DW")
print(f"  {'Model 1':<13} {int(m1.nobs):>5}   {m1.rsquared:.4f}  {m1.rsquared_adj:.4f}   {durbin_watson(m1.resid):.3f}")
print(f"  {'Model 2 (AR)':<13} {int(m2.nobs):>5}   {m2.rsquared:.4f}  {m2.rsquared_adj:.4f}   {durbin_watson(m2.resid):.3f}")
print(f"  {'Model 3a':<13} {int(m3a.nobs):>5}   {m3a.rsquared:.4f}  {m3a.rsquared_adj:.4f}   {durbin_watson(m3a_raw.resid):.3f}")
print(f"  {'Model 3b':<13} {int(m3b.nobs):>5}   {m3b.rsquared:.4f}  {m3b.rsquared_adj:.4f}   {durbin_watson(m3b_raw.resid):.3f}")
print()
print("  Saved outputs:")
print(f"    intermediary/high_resource_countries.csv")
print("=" * 70)

## Section 5 — Regression & Diagnostic Charts: 15, 17, 26, 27–34

Charts from viz_updates.py (improved styling) and from the ML/regression pipelines:

- **15**: ECI cluster trajectories (median by resource profile)
- **17**: HCI × Production interaction scatter
- **26**: PCA resource loadings heatmap
- **27**: Random Forest feature importance (from NB5 RF model)
- **28**: Variance inflation factors
- **29**: Bootstrap R² and coefficient stability
- **31**: ML prediction intervals (by country)
- **32**: Country data coverage scatter
- **33**: Full variable correlation matrix

**Note:** Charts 27, 28, 29, 33, 34 require running the bootstrap pipeline (`scripts/run_bootstrap.py`) to produce files in `intermediary/bootstrap/`. If those files are missing, those charts will be skipped.

In [ ]:
# Restore NB5 output path for viz_updates compatibility
OUT = 'Final/NB5'
NB5 = OUT

# Restore CLUSTER_COLORS as a dict (viz_updates uses it as a dict)
# viz_utils provides CLUSTER_COLORS as a dict keyed by cluster_id
# Some viz_updates code expects integer keys

# CHART 15 — ECI Cluster Trajectories (cluster names instead of numbers)
# =============================================================================
print("\n[15] ECI Cluster Trajectories — cluster names...")

master   = load_master()
clusters = load_clusters('1995')[['Country Code', 'Cluster']].drop_duplicates()
df       = master[master['Country Code'].isin(INCLUDE_LIST)].copy()
df       = df.merge(clusters, on='Country Code', how='left')

traj = (df.groupby(['Year', 'Cluster'])['Economic Complexity Index']
          .median().reset_index())

fig = go.Figure()
for cl in sorted(traj['Cluster'].dropna().unique()):
    sub = traj[traj['Cluster'] == cl]
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers',
        name=CLUSTER_LABELS.get(int(cl), f'Cluster {int(cl)}'),
        line=dict(color=CLUSTER_COLORS.get(int(cl), '#999'), width=2.2),
        marker=dict(size=5),
        hovertemplate='%{x}: %{y:.3f}<extra>' +
                      CLUSTER_LABELS.get(int(cl), '') + '</extra>',
    ))

fig.update_layout(**base_layout(
    height=480,
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
    hovermode='x unified',
))
save(fig, '15_reg__eci_mean_trajectory_by_cluster', OUT)


# =============================================================================
# CHART 16 — Coefficients 3a vs 3b (remove Driscoll-Kraay from axis title)
# =============================================================================
print("\n[16] Coefficients 3a vs 3b — removing Driscoll-Kraay...")

chart16_path = os.path.join(OUT, '16_reg__coefficients_model3a_vs_model3b.html')
if os.path.exists(chart16_path):
    html = open(chart16_path, encoding='utf-8').read()
    html = html.replace('95 % CI, Driscoll-Kraay', '95% CI')
    html = html.replace('95% CI, Driscoll-Kraay', '95% CI')
    html = html.replace('Driscoll-Kraay', '')
    open(chart16_path, 'w', encoding='utf-8').write(html)
    print(f"  Updated: 16_reg__coefficients_model3a_vs_model3b.html")


# =============================================================================
# CHART 17 — HCI × Production interaction (simplified: country means)
# =============================================================================
print("\n[17] HCI × Production interaction — simplifying...")

master = load_master()
df     = master[master['Country Code'].isin(INCLUDE_LIST)].copy()
df['prod_pc']    = df['Total_Production_Value'] / df['Population'].replace(0, np.nan)
df['log_HCI']    = np.log1p(df['Human capital index'])
df['log_prod_pc']= np.log1p(df['prod_pc'])

# Aggregate to country means — one point per country, much cleaner than all obs
country_avg = (
    df[['Country Code', 'log_HCI', 'Economic Complexity Index', 'log_prod_pc']]
    .dropna()
    .groupby('Country Code')
    .mean()
    .reset_index()
)

# Assign production quartile based on country average
country_avg['Prod_quartile'] = pd.qcut(
    country_avg['log_prod_pc'], q=4,
    labels=['Q1 — Low production', 'Q2', 'Q3', 'Q4 — High production']
)

q_colors = [PALETTE['light_blue'], PALETTE['blue'], PALETTE['orange'], PALETTE['red']]

fig = go.Figure()
for q, col in zip(['Q1 — Low production', 'Q2', 'Q3', 'Q4 — High production'], q_colors):
    sub = country_avg[country_avg['Prod_quartile'] == q]
    fig.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub['Economic Complexity Index'],
        mode='markers+text',
        text=sub['Country Code'],
        textposition='top center',
        textfont=dict(size=8, color='#555'),
        marker=dict(color=col, size=9, opacity=0.85,
                    line=dict(width=0.8, color='white')),
        name=f'Production {q}',
        hovertemplate='<b>%{text}</b><br>log(HCI): %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
    ))

fig.update_layout(**base_layout(
    height=540,
    xaxis=dict(title='log(Human Capital Index)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5),
    legend=dict(title=dict(text='Avg. Production p.c. Quartile'),
                font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig, '17_reg__hci_production_interaction_effect_on_eci', OUT)


# =============================================================================
# CHART 26 — PCA Resource Loadings Heatmap
#   - Red-white-blue colorscale (was yellow-white-blue)
#   - PC1 row at top
#   - Larger colorbar and chart
# =============================================================================
print("\n[26] PCA Loadings heatmap — fixing colours, PC1 first, larger...")

nr        = load_nr()
nr_sample = nr[nr['Country Code'].isin(INCLUDE_LIST)]
nr_1995   = nr_sample[nr_sample['Year'] == 1995]

pivot = nr_1995.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],
    columns='Resource', values='Production_TotalValue',
).reset_index()

resource_cols = [c for c in pivot.columns
                 if c not in ['Country', 'Country Code', 'Year', 'Population']]
pivot[resource_cols] = pivot[resource_cols].div(pivot['Population'], axis=0)
pivot = pivot.fillna(0)

X   = np.log1p(pivot[resource_cols].fillna(0))
pca = PCA(n_components=2, random_state=42)
pca.fit(X)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=resource_cols)
top20    = loadings.abs().sum(axis=1).nlargest(20).index
plot_df  = loadings.loc[top20]
plot_df  = (plot_df
            .assign(_s=plot_df['PC1'].abs() + plot_df['PC2'].abs())
            .sort_values('_s', ascending=False)
            .drop(columns='_s'))

# PC labels — PC1 first = top row (reversed y in Plotly)
# PC1 loads on oil & gas (hydrocarbons); PC2 loads on copper, gold & coal
pc_labels_ordered = [
    f'PC1 ({var1:.1f}%)<br><i>↑ Oil & Gas</i>',
    f'PC2 ({var2:.1f}%)<br><i>↑ Copper, Gold & Coal</i>',
]
# Plotly heatmap y goes bottom→top, so put PC2 first in the list → PC1 on top
y_labels = pc_labels_ordered[::-1]          # [PC2, PC1] → displayed bottom→top = PC2 bottom, PC1 top
z_values = plot_df[['PC2', 'PC1']].T.values  # rows match y_labels order

fig = go.Figure(go.Heatmap(
    z=z_values,
    x=plot_df.index.tolist(),
    y=y_labels,
    colorscale=[
        [0.0, '#c23a3a'],
        [0.5, '#ffffff'],
        [1.0, '#1a4a8a'],
    ],
    zmid=0, zmin=-1, zmax=1,
    text=z_values.round(2),
    texttemplate='%{text:.2f}',
    textfont=dict(size=8, family=FONT),
    hovertemplate='%{x} / %{y}: %{z:.3f}<extra></extra>',
    colorbar=dict(
        title=dict(text='Loading', font=dict(size=12)),
        thickness=20, len=1.0,
        tickvals=[-1, -0.5, 0, 0.5, 1],
        tickfont=dict(size=11),
    ),
))

fig.update_xaxes(title_text='Resource/Feature', tickangle=-40, tickfont=dict(size=10, family=FONT), showgrid=False)
fig.update_yaxes(title_text='Principal Component', tickfont=dict(size=12, family=FONT), showgrid=False)
fig.update_layout(**base_layout(
    height=420,
    margin=dict(l=260, r=120, t=60, b=160),
))
save(fig, '26_diag__pca_resource_loadings_heatmap', OUT, w=1300, h=420)


# =============================================================================
# CHART 31 — ML Prediction Intervals (aggregated by country)
# =============================================================================
print("\n[31] Prediction intervals — aggregating by country...")

preds = pd.read_csv(os.path.join(NB5, 'test_predictions.csv'))

# Country-level: mean actual, mean predicted, std actual over test years
country_stats = (
    preds.groupby(['Country Code', 'Country Name'])
    .agg(
        Actual_mean   = ('Actual_ECI', 'mean'),
        Predicted_mean= ('Predicted_ECI', 'mean'),
        Actual_std    = ('Actual_ECI', 'std'),
        n_years       = ('Year', 'count'),
    )
    .reset_index()
    .sort_values('Actual_mean')
    .reset_index(drop=True)
)
country_stats['Actual_std'] = country_stats['Actual_std'].fillna(0)

# In-band: |actual_mean - predicted_mean| < 1 std dev
country_stats['In_band'] = (
    (country_stats['Actual_mean'] - country_stats['Predicted_mean']).abs()
    < country_stats['Actual_std']
)

fig = go.Figure()

# Std-dev band per country (horizontal error bar on actual)
fig.add_trace(go.Scatter(
    x=list(range(len(country_stats))) * 2 + list(range(len(country_stats)))[::-1] * 2,
    y=(country_stats['Actual_mean'] + country_stats['Actual_std']).tolist() +
      (country_stats['Actual_mean'] - country_stats['Actual_std']).iloc[::-1].tolist(),
    fill='toself', fillcolor='rgba(74,111,165,0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    hoverinfo='skip', name='±1 SD (actual ECI in test years)',
))

fig.add_trace(go.Scatter(
    x=list(range(len(country_stats))),
    y=country_stats['Actual_mean'],
    mode='lines', line=dict(color=PALETTE['blue'], width=2),
    name='Mean Actual ECI',
))

for in_band, color, sym, lbl in [
    (True,  PALETTE['green'], 'circle',  'Predicted ≈ Actual (within ±1 SD)'),
    (False, PALETTE['red'],   'diamond', 'Predicted outside ±1 SD'),
]:
    mask = country_stats['In_band'] == in_band
    sub  = country_stats[mask]
    fig.add_trace(go.Scatter(
        x=sub.index.tolist(),
        y=sub['Predicted_mean'],
        mode='markers',
        marker=dict(color=color, size=8 if in_band else 10, symbol=sym, opacity=0.85),
        name=lbl,
        customdata=sub[['Country Code', 'Country Name']].values,
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]})<br>'
                      'Avg Actual: %{text}<br>Avg Predicted: %{y:.3f}<extra></extra>',
        text=[f'{v:.3f}' for v in sub['Actual_mean']],
    ))

fig.update_layout(**base_layout(
    height=500,
    xaxis=dict(
        title='Countries (sorted by mean actual ECI)',
        tickvals=list(range(len(country_stats))),
        ticktext=country_stats['Country Code'].tolist(),
        tickangle=-60, tickfont=dict(size=8),
        gridcolor=GRID,
    ),
    yaxis=dict(title='ECI (test set mean)', gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig, '31_diag__ml_prediction_intervals', OUT)


# =============================================================================
# CHART 32 — Country Data Coverage (highlight high-missingness countries)
# =============================================================================
print("\n[32] Country data coverage — highlighting high-missingness countries...")

raw = load_master_wide()
sample_df = raw[raw['Country Code'].isin(INCLUDE_LIST)].copy()
country_missing = analyze_country_missingness(sample_df)

LABEL_THRESHOLD = 20.0   # label countries above this % missing

fig = go.Figure()

for above in [True, False]:
    mask = country_missing['% Missing'] >= LABEL_THRESHOLD if above else \
           country_missing['% Missing'] < LABEL_THRESHOLD
    sub  = country_missing[mask]
    color= PALETTE['red'] if above else PALETTE['blue']
    size = 12 if above else 8
    mode = 'markers+text' if above else 'markers'

    fig.add_trace(go.Scatter(
        x=sub['Vars with Data'],
        y=sub['% Missing'],
        mode=mode,
        text=sub['Code'] if above else None,
        textposition='top center',
        textfont=dict(size=9, color=PALETTE['red']),
        marker=dict(color=color, size=size, opacity=0.75 if above else 0.55,
                    line=dict(color='white', width=0.8)),
        name=f'>= {LABEL_THRESHOLD}% missing' if above else f'< {LABEL_THRESHOLD}% missing',
        customdata=sub[['Code', 'Country', 'Complete Vars', 'Years Covered', 'Rows']].values,
        hovertemplate=(
            '<b>%{customdata[1]}</b> (%{customdata[0]})<br>'
            'Vars with data: %{x}<br>'
            '% Missing: %{y:.1f}%<br>'
            'Complete vars: %{customdata[2]}<br>'
            'Years covered: %{customdata[3]}<extra></extra>'
        ),
    ))

med_vars    = country_missing['Vars with Data'].median()
med_missing = country_missing['% Missing'].median()
fig.add_hline(y=med_missing, line_dash='dash', line_color='#aaa', opacity=0.6,
              annotation_text=f'Median {med_missing:.1f}%', annotation_position='right')
fig.add_vline(x=med_vars, line_dash='dash', line_color='#aaa', opacity=0.6,
              annotation_text=f'Median {med_vars:.0f} vars', annotation_position='top')

fig.update_layout(**base_layout(
    height=520,
    xaxis=dict(title='Variables with Any Data', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='% Missing Data Overall', gridcolor=GRID, gridwidth=0.5),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig, '32_diag__country_data_coverage_scatter', OUT)


# =============================================================================
# CHART 27 — Random Forest Feature Importance (from NB5 all_importance.csv)
# =============================================================================
print("\n[27] Random Forest feature importance...")
try:
    _imp27 = pd.read_csv(os.path.join(NB5, 'all_importance.csv'))
    if 'Random Forest' in _imp27.columns:
        _rf27 = (_imp27[_imp27['Feature'] != 'L1_ECI']
                 .sort_values('Random Forest', ascending=False)
                 .head(15).reset_index(drop=True))
        _rf27 = _rf27.iloc[::-1].reset_index(drop=True)

        def _lerp_color27(t, c_lo=(230,185,128), c_hi=(180,80,40)):
            return f'rgb({int(c_lo[0]+(c_hi[0]-c_lo[0])*t)},{int(c_lo[1]+(c_hi[1]-c_lo[1])*t)},{int(c_lo[2]+(c_hi[2]-c_lo[2])*t)})'

        _norm27 = (_rf27['Random Forest'] / _rf27['Random Forest'].max()).values
        _colors27 = [_lerp_color27(v) for v in _norm27]

        _fig27 = go.Figure(go.Bar(
            y=[shorten_feat(f) for f in _rf27['Feature']],
            x=_rf27['Random Forest'], orientation='h',
            marker=dict(color=_colors27, line=dict(color=NAVY, width=0.5)),
            text=[f'{v:.4f}' for v in _rf27['Random Forest']], textposition='outside',
            textfont=dict(size=11, color=NAVY),
        ))
        _fig27.update_layout(**base_layout(
            height=STYLE['chart_height_tall'],
            margin=dict(l=200, r=80, t=70, b=60),
            xaxis=dict(title=dict(text='Mean Decrease in Impurity', font=dict(size=13)),
                       gridcolor=GRID, gridwidth=0.5,
                       range=[0, _rf27['Random Forest'].max()*1.22]),
            yaxis=dict(tickfont=dict(size=11)),
            showlegend=False,
        ))
        save(_fig27, '27_diag__random_forest_feature_importance', OUT_CHARTS, w=1100, h=700)
    else:
        print("  Random Forest column not found in all_importance.csv")
except Exception as e:
    print(f"  [27] Skipped: {e}")


# =============================================================================
# CHART 28 — VIF Diagnostics
# =============================================================================
print("\n[28] VIF diagnostics...")
try:
    _vif28 = pd.read_csv(os.path.join(NB5, 'coefficient_summary_table.csv'))
    if 'VIF' in _vif28.columns:
        _vif28 = _vif28.sort_values('VIF', ascending=True).reset_index(drop=True)
        _vif28['Short'] = _vif28['Feature'].apply(shorten_feat)
        _colors28 = [PALETTE['red'] if v > 10 else PALETTE['blue'] for v in _vif28['VIF']]

        _fig28 = go.Figure(go.Bar(
            y=_vif28['Short'], x=_vif28['VIF'], orientation='h',
            marker=dict(color=_colors28, line=dict(color=NAVY, width=0.5)),
            text=[f'{v:.1f}' for v in _vif28['VIF']], textposition='outside',
            textfont=dict(size=11, color=NAVY),
        ))
        _fig28.add_vline(x=10, line=dict(color=PALETTE['red'], width=2, dash='dash'),
                         annotation_text='VIF = 10', annotation_position='right',
                         annotation_font=dict(size=11, color=PALETTE['red']))
        _fig28.add_vline(x=5, line=dict(color=PALETTE['orange'], width=1.5, dash='dot'),
                         annotation_text='VIF = 5', annotation_position='right',
                         annotation_font=dict(size=11, color=PALETTE['orange']))
        _fig28.update_layout(**base_layout(
            height=STYLE['chart_height_tall'],
            margin=dict(l=200, r=120, t=70, b=60),
            xaxis=dict(title=dict(text='Variance Inflation Factor', font=dict(size=13)),
                       gridcolor=GRID, gridwidth=0.5,
                       range=[0, max(_vif28['VIF'].max()*1.15, 12)]),
            yaxis=dict(tickfont=dict(size=11)),
            showlegend=False,
        ))
        save(_fig28, '28_diag__variance_inflation_factors', OUT_CHARTS, w=1100, h=700)
    else:
        print("  VIF column not found")
except Exception as e:
    print(f"  [28] Skipped: {e}")


# =============================================================================
# CHART 29 — Bootstrap R² and Coefficient Stability
# =============================================================================
print("\n[29] Bootstrap R² and coefficient stability...")
_boot_path = os.path.join('intermediary', 'bootstrap')
try:
    _boot_files = [f for f in os.listdir(_boot_path) if f.endswith('.csv')] if os.path.isdir(_boot_path) else []
    if not _boot_files:
        print("  [29] Skipped — no bootstrap files in intermediary/bootstrap/ (run run_bootstrap.py first)")
    else:
        # Try to load bootstrap regression results
        _boot_reg = None
        for fn in ['bootstrap_regression_results.csv', 'bootstrap_nb6.csv', 'bootstrap_results.csv']:
            fp = os.path.join(_boot_path, fn)
            if os.path.exists(fp):
                _boot_reg = pd.read_csv(fp)
                break

        if _boot_reg is not None:
            # Plot distribution of R² across bootstrap samples
            _r2_col = [c for c in _boot_reg.columns if 'r2' in c.lower() or 'R2' in c or 'R²' in c]
            if _r2_col:
                _fig29 = go.Figure()
                for col in _r2_col[:4]:
                    _fig29.add_trace(go.Histogram(
                        x=_boot_reg[col], name=col, opacity=0.75,
                        nbinsx=30,
                        marker=dict(line=dict(color='white', width=0.5)),
                    ))
                _fig29.update_layout(**base_layout(
                    height=500, barmode='overlay',
                    xaxis=dict(title='Bootstrap R²', gridcolor=GRID),
                    yaxis=dict(title='Count', gridcolor=GRID),
                    legend=dict(font=dict(size=10)),
                ))
                save(_fig29, '29_diag__bootstrap_r2_and_coefficient_stability', OUT_CHARTS, w=1100, h=500)
            else:
                print("  [29] No R² column found in bootstrap file")
        else:
            print("  [29] Bootstrap regression file not found — run run_bootstrap.py first")
except Exception as e:
    print(f"  [29] Skipped: {e}")


# =============================================================================
# CHART 33 — Full Variable Correlation Matrix
# =============================================================================
print("\n[33] Full variable correlation matrix...")
try:
    _master33 = load_master()
    _df33 = _master33[_master33['Country Code'].isin(INCLUDE_LIST)].copy()
    # Select numeric columns with enough coverage
    _num_cols = _df33.select_dtypes(include=[np.number]).columns.tolist()
    _excl = ['Year', 'Population', 'Unnamed: 0']
    _num_cols = [c for c in _num_cols if c not in _excl and _df33[c].notna().mean() > 0.5]
    _corr33 = _df33[_num_cols].corr().round(2)
    # Limit to 25 most connected variables
    _conn = _corr33.abs().sum(axis=1).nlargest(25).index
    _corr33 = _corr33.loc[_conn, _conn]

    _labels33 = [shorten_feat(c, max_len=30) for c in _corr33.columns]
    _z33 = _corr33.values

    _fig33 = go.Figure(go.Heatmap(
        z=_z33, x=_labels33, y=_labels33,
        colorscale=[[0.0, PALETTE['red']], [0.5, '#fafafa'], [1.0, PALETTE['blue']]],
        zmid=0, zmin=-1, zmax=1,
        hovertemplate='%{x} × %{y}: %{z:.2f}<extra></extra>',
        colorbar=dict(thickness=14, len=0.9, tickfont=dict(size=11)),
    ))
    _fig33.update_layout(**base_layout(
        height=800,
        margin=dict(l=220, r=60, t=60, b=220),
        xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9)),
    ))
    save(_fig33, '33_diag__full_variable_correlation_matrix', OUT_CHARTS, w=1000, h=800)
except Exception as e:
    print(f"  [33] Skipped: {e}")


# =============================================================================
# CHART 34 — HRV Growth Diagnostics
# =============================================================================
print("\n[34] HRV growth diagnostics...")
try:
    _master34 = load_master()
    _df34 = _master34[_master34['Country Code'].isin(INCLUDE_LIST)].copy()

    # Compute ECI growth rate per country
    _df34 = _df34.sort_values(['Country Code', 'Year'])
    _df34['ECI_lag1'] = _df34.groupby('Country Code')['Economic Complexity Index'].shift(1)
    _df34['ECI_growth'] = _df34['Economic Complexity Index'] - _df34['ECI_lag1']

    # Production value growth
    if 'Total_Production_Value' in _df34.columns and 'Population' in _df34.columns:
        _df34['prod_pc'] = _df34['Total_Production_Value'] / _df34['Population'].replace(0, np.nan)
        _df34['prod_lag1'] = _df34.groupby('Country Code')['prod_pc'].shift(1)
        _df34['prod_growth'] = np.log(_df34['prod_pc'] / _df34['prod_lag1'].replace(0, np.nan))

        _plot34 = _df34[['Country Code', 'Year', 'ECI_growth', 'prod_growth',
                          'Human capital index']].dropna()

        _fig34 = go.Figure()
        _fig34.add_trace(go.Scatter(
            x=_plot34['prod_growth'],
            y=_plot34['ECI_growth'],
            mode='markers',
            marker=dict(
                size=6, opacity=0.45, color=PALETTE['blue'],
                line=dict(color='white', width=0.3),
            ),
            hovertemplate='%{customdata[0]} %{customdata[1]}<br>Prod growth: %{x:.3f}<br>ECI growth: %{y:.3f}<extra></extra>',
            customdata=_plot34[['Country Code', 'Year']].values,
            name='Country-year obs.',
        ))
        # Trend line
        _valid = _plot34.dropna(subset=['prod_growth', 'ECI_growth'])
        if len(_valid) > 10:
            _z34 = np.polyfit(_valid['prod_growth'], _valid['ECI_growth'], 1)
            _x34 = np.linspace(_valid['prod_growth'].min(), _valid['prod_growth'].max(), 100)
            _fig34.add_trace(go.Scatter(
                x=_x34, y=np.polyval(_z34, _x34),
                mode='lines', line=dict(color=PALETTE['red'], width=2, dash='dash'),
                name=f'OLS trend (slope={_z34[0]:.3f})',
            ))
        _fig34.update_layout(**base_layout(
            height=500,
            xaxis=dict(title='Log Production Value p.c. Growth', gridcolor=GRID, gridwidth=0.5),
            yaxis=dict(title='ECI Year-on-Year Change', gridcolor=GRID, gridwidth=0.5,
                       zeroline=True, zerolinecolor='#ddd'),
            legend=dict(font=dict(size=10)),
        ))
        save(_fig34, '34_diag__hrv_growth_diagnostics_framework', OUT_CHARTS, w=1100, h=500)
    else:
        print("  [34] Required columns not found")
except Exception as e:
    print(f"  [34] Skipped: {e}")

## Section 6 — Case Studies: Charts 21–24

These charts present case study analysis for Congo (COG), Azerbaijan (AZE), and Chile (CHL).

In [ ]:
# =============================================================================
# CHARTS 21–24 — Case Study Charts
# =============================================================================
# Case study charts for Congo (COG), Azerbaijan (AZE), and Chile (CHL).
# These charts compare country-specific trajectories against cluster peers.

print("\n[21–24] Case study charts...")
try:
    _master_cs = load_master()
    _clusters_cs = load_clusters('1995')[['Country Code', 'Cluster']].drop_duplicates()
    _df_cs = _master_cs[_master_cs['Country Code'].isin(INCLUDE_LIST)].copy()
    _df_cs = _df_cs.merge(_clusters_cs, on='Country Code', how='left')

    # ── CHART 21: Congo Macro Indicators vs Sample ────────────────────────────
    print("  [21] Congo macro indicators vs sample...")
    CASE_CC = 'COG'
    CASE_NAME = 'Congo'
    _macro_vars = {
        'Human capital index': 'Human Capital Index',
        'Rule of law index': 'Rule of Law Index',
        'Trade (% of GDP)': 'Trade (% GDP)',
        'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'GFCF (% GDP)',
        'Access to electricity (% of population)': 'Electricity Access (%)',
    }
    _case_df = _df_cs[_df_cs['Country Code'] == CASE_CC].sort_values('Year')
    _sample_yr = _df_cs.groupby('Year')[list(_macro_vars.keys())].median()

    _fig21 = make_subplots(rows=2, cols=3,
                           subplot_titles=list(_macro_vars.values()),
                           vertical_spacing=0.15, horizontal_spacing=0.10)

    for i, (var, label) in enumerate(_macro_vars.items()):
        row_n = i // 3 + 1
        col_n = i % 3 + 1
        if var not in _case_df.columns:
            continue
        _fig21.add_trace(go.Scatter(
            x=_sample_yr.index, y=_sample_yr[var] if var in _sample_yr else [],
            mode='lines', name='Sample median' if i == 0 else '',
            showlegend=(i == 0),
            line=dict(color=PALETTE['blue'], width=1.5, dash='dash'),
            legendgroup='median',
        ), row=row_n, col=col_n)
        _fig21.add_trace(go.Scatter(
            x=_case_df['Year'], y=_case_df[var],
            mode='lines+markers', name=CASE_NAME if i == 0 else '',
            showlegend=(i == 0),
            line=dict(color=PALETTE['red'], width=2.2),
            marker=dict(size=4),
            legendgroup='case',
        ), row=row_n, col=col_n)
        _fig21.update_xaxes(gridcolor=GRID, gridwidth=0.5, row=row_n, col=col_n)
        _fig21.update_yaxes(gridcolor=GRID, gridwidth=0.5, row=row_n, col=col_n)

    _fig21.update_layout(**base_layout(
        height=600, margin=dict(l=60, r=40, t=100, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='center', x=0.5, font=dict(size=11)),
    ))
    save(_fig21, '21_case__congo_macro_indicators_vs_sample', OUT_CHARTS, w=1200, h=600)

    # ── CHART 22: Congo ECI Trajectory vs Cluster Peers ──────────────────────
    print("  [22] Congo ECI trajectory vs cluster peers...")
    _cog_cluster = int(_clusters_cs[_clusters_cs['Country Code'] == CASE_CC]['Cluster'].values[0])         if len(_clusters_cs[_clusters_cs['Country Code'] == CASE_CC]) else None

    _fig22 = go.Figure()
    if _cog_cluster is not None:
        _peers = _clusters_cs[_clusters_cs['Cluster'] == _cog_cluster]['Country Code'].tolist()
        for cc in _peers:
            if cc == CASE_CC:
                continue
            _peer_df = _df_cs[_df_cs['Country Code'] == cc].sort_values('Year')
            _cname = _peer_df['Country Name'].iloc[0] if 'Country Name' in _peer_df.columns and len(_peer_df) else cc
            _fig22.add_trace(go.Scatter(
                x=_peer_df['Year'], y=_peer_df['Economic Complexity Index'],
                mode='lines', line=dict(color='#b0b8c4', width=0.8), opacity=0.4,
                name='Cluster peers' if cc == _peers[0] else '',
                showlegend=(cc == _peers[0]),
                legendgroup='peers',
            ))

    _fig22.add_trace(go.Scatter(
        x=_case_df['Year'], y=_case_df['Economic Complexity Index'],
        mode='lines+markers', name=CASE_NAME,
        line=dict(color=PALETTE['red'], width=2.5),
        marker=dict(size=5),
    ))
    _fig22.update_layout(**base_layout(
        height=480,
        xaxis=dict(title='Year', gridcolor=GRID, dtick=5),
        yaxis=dict(title='Economic Complexity Index', gridcolor=GRID,
                   zeroline=True, zerolinecolor='#ddd'),
        legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                    bordercolor=GRID, borderwidth=1),
    ))
    save(_fig22, '22_case__congo_eci_trajectory_vs_cluster_peers', OUT_CHARTS, w=1100, h=480)

    # ── CHART 23: Azerbaijan ECI Trajectory vs Cluster Peers ─────────────────
    print("  [23] Azerbaijan ECI trajectory vs cluster peers...")
    AZE_CC = 'AZE'
    _aze_df = _df_cs[_df_cs['Country Code'] == AZE_CC].sort_values('Year')
    _aze_cluster = int(_clusters_cs[_clusters_cs['Country Code'] == AZE_CC]['Cluster'].values[0])         if len(_clusters_cs[_clusters_cs['Country Code'] == AZE_CC]) else None

    _fig23 = go.Figure()
    if _aze_cluster is not None:
        _peers23 = _clusters_cs[_clusters_cs['Cluster'] == _aze_cluster]['Country Code'].tolist()
        for cc in _peers23:
            if cc == AZE_CC:
                continue
            _peer_df = _df_cs[_df_cs['Country Code'] == cc].sort_values('Year')
            _fig23.add_trace(go.Scatter(
                x=_peer_df['Year'], y=_peer_df['Economic Complexity Index'],
                mode='lines', line=dict(color='#b0b8c4', width=0.8), opacity=0.4,
                name='Cluster peers' if cc == _peers23[0] else '',
                showlegend=(cc == _peers23[0]),
                legendgroup='peers23',
            ))

    _fig23.add_trace(go.Scatter(
        x=_aze_df['Year'], y=_aze_df['Economic Complexity Index'],
        mode='lines+markers', name='Azerbaijan',
        line=dict(color=PALETTE['orange'], width=2.5),
        marker=dict(size=5),
    ))
    _fig23.update_layout(**base_layout(
        height=480,
        xaxis=dict(title='Year', gridcolor=GRID, dtick=5),
        yaxis=dict(title='Economic Complexity Index', gridcolor=GRID,
                   zeroline=True, zerolinecolor='#ddd'),
        legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                    bordercolor=GRID, borderwidth=1),
    ))
    save(_fig23, '23_case__azerbaijan_eci_trajectory_vs_cluster_peers', OUT_CHARTS, w=1100, h=480)

    # ── CHART 24: ECI Forecast 2020–2030 — Three Countries ───────────────────
    print("  [24] ECI forecast — three countries...")
    _fc_path = os.path.join('Final', 'NB5', 'ECI_Forecast_2020_2030.csv')
    if os.path.exists(_fc_path):
        _fc24 = pd.read_csv(_fc_path)
        _case_codes = ['COG', 'AZE', 'CHL']
        _case_colors = [PALETTE['red'], PALETTE['orange'], PALETTE['blue']]
        _case_names  = ['Congo', 'Azerbaijan', 'Chile']

        _fig24 = go.Figure()
        for cc, col, name in zip(_case_codes, _case_colors, _case_names):
            _hist24 = _df_cs[_df_cs['Country Code'] == cc].sort_values('Year')
            _fore24 = _fc24[_fc24['Country Code'] == cc].sort_values('Year')
            if _hist24.empty or _fore24.empty:
                continue
            _last_yr  = int(_hist24['Year'].iloc[-1])
            _last_eci = float(_hist24['Economic Complexity Index'].iloc[-1])
            _fig24.add_trace(go.Scatter(
                x=_hist24['Year'], y=_hist24['Economic Complexity Index'],
                mode='lines', name=name,
                line=dict(color=col, width=2.2),
                legendgroup=cc,
            ))
            _fig24.add_trace(go.Scatter(
                x=[_last_yr] + _fore24['Year'].tolist(),
                y=[_last_eci] + _fore24['Ensemble'].tolist(),
                mode='lines', line=dict(color=col, width=2.2, dash='dash'),
                legendgroup=cc, showlegend=False,
            ))
        _fig24.add_vline(x=2019.5, line=dict(color='#aaa', width=1.5, dash='dot'))
        _fig24.update_layout(**base_layout(
            height=520,
            xaxis=dict(title='Year', gridcolor=GRID, dtick=5),
            yaxis=dict(title='Economic Complexity Index', gridcolor=GRID,
                       zeroline=True, zerolinecolor='#ddd'),
            legend=dict(font=dict(size=11), bgcolor='rgba(255,255,255,0.9)',
                        bordercolor=GRID, borderwidth=1),
            annotations=[dict(
                x=2020, y=0.02, xref='paper', yref='paper',
                text='← Historical | Forecast →', showarrow=False,
                font=dict(size=10, color='#888'),
            )],
        ))
        save(_fig24, '24_case__eci_forecast_2020_2030_three_countries', OUT_CHARTS, w=1100, h=520)
    else:
        print("  [24] Skipped — ECI_Forecast_2020_2030.csv not found (run Section 2 first)")

    print("\n[21–24] Case study charts complete.")

except Exception as e:
    import traceback
    print(f"  Case study charts failed: {e}")
    traceback.print_exc()